# Backpropagation From Scratch — Complete Derivations (Colab)

**This notebook is fully self-contained.** Every code cell uses only `numpy` and `math`, which are pre-installed on Google Colab — there is **nothing to `pip install`, no Drive to mount, and no GPU required**.

**How to run:** `Runtime → Run all` (or run each cell top to bottom). The numerical-gradient checks in Section 4.3 will print errors around `1e-10`, confirming every analytic formula matches reality.

> Tip: Section 4.3 is the heart of the matrix-calculus walkthrough — the "three facts", the fan-out→sum rule, the three-way numerical proof, and the auto-checked tiny network.

# Backpropagation From Scratch — A Complete, Beginner-Friendly Derivation

**What this is.** This is my own self-contained, step-by-step derivation of **forward propagation** and **backward propagation** (backprop), built up from the simplest possible model to a general deep network — and then to the common architectures (softmax classifiers, CNNs, RNNs).

I derive everything **by hand first** (with all the algebra shown), then implement it in **plain NumPy**, then **verify it numerically** so I can trust every formula (and so can you).

> **Anchor.** Section 4 reproduces *exactly* the network from `03_backprop.ipynb` of the fast.ai course — a one-hidden-layer net `784 → 50 → 1` with ReLU and MSE loss — and matches its `lin_grad` / `forward_and_backward` code line-for-line in NumPy.

### Roadmap (increasing complexity)

| Section | Model | New idea introduced |
|---|---|---|
| 0 | — | Notation + the **chain rule**, the only tool we need |
| 1 | Building blocks | Linear layer, activations, losses, and **their derivatives** |
| 2 | **Linear neuron** (no hidden layer) | The simplest forward/backward pair |
| 3 | **Logistic regression** (sigmoid + BCE) | Where the famous `(a − y)` comes from |
| 4 | **One hidden layer** (the `03_backprop` net) | Stacking layers → the chain rule across layers |
| 5 | **Deep net, L layers** (vectorized, batched) | The **four equations of backprop** in full generality |
| 6 | **Softmax + cross-entropy** | Multi-class classification (`δ = a − y`) |
| 7 | **CNN** convolutional layer | Backprop through a convolution |
| 8 | **RNN** (vanilla) | Backprop **through time** (BPTT) |
| 9 | Cheat-sheet | All equations on one page |

**How to read.** Run the cells top to bottom. Each model ends with a numerical gradient check printing `OK`. If you only want the intuition, read the markdown; if you want proof, read the code.

## 0. Notation and the one tool you need: the chain rule

### 0.1 Notation

We use the standard "layered" notation. For a network with $L$ layers:

- $\mathbf{x}$ — input (also written $\mathbf{a}^{[0]}$).
- $W^{[l]}, \mathbf{b}^{[l]}$ — **weights** and **bias** of layer $l$.
- $\mathbf{z}^{[l]} = W^{[l]}\mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$ — the **pre-activation** (a linear combination).
- $\mathbf{a}^{[l]} = g^{[l]}(\mathbf{z}^{[l]})$ — the **activation** (apply a nonlinearity $g$ elementwise).
- $\hat{\mathbf{y}} = \mathbf{a}^{[L]}$ — the network's **output / prediction**.
- $L(\hat{\mathbf{y}}, \mathbf{y})$ — the **loss** (a single number measuring "how wrong").

A central shorthand we will use everywhere:

$$\boxed{\;\boldsymbol{\delta}^{[l]} \;\equiv\; \frac{\partial L}{\partial \mathbf{z}^{[l]}}\;}$$

$\boldsymbol{\delta}^{[l]}$ is the **error signal** at layer $l$ — the gradient of the loss with respect to that layer's pre-activation. Almost all of backprop is: *"compute $\boldsymbol{\delta}$ at the output, then push it backwards layer by layer."*

> **Notation note.** Different books transpose things differently. We use the **machine-learning / NumPy convention** where a data batch $X$ has shape `(n_samples, n_features)` — i.e. one example *per row*. This is exactly the convention in `03_backprop.ipynb`, so the matrix shapes below will match that notebook.

### 0.2 The chain rule — the entire secret of backprop

If $L$ depends on $u$, and $u$ depends on $w$, then

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial u}\,\frac{\partial u}{\partial w}.$$

When $u$ is a **vector** $\mathbf{u}=(u_1,\dots,u_k)$ and each $u_i$ depends on $w$, the contributions **add up**:

$$\frac{\partial L}{\partial w} = \sum_{i=1}^{k} \frac{\partial L}{\partial u_i}\,\frac{\partial u_i}{\partial w}.$$

That "sum over every path from $w$ to the loss" is the *only* idea in backprop. A neural network is just a long composition of functions $L\big(g(W\cdots)\big)$; backprop is the chain rule applied to that composition, **reusing** shared sub-results (the $\boldsymbol{\delta}$'s) so we never recompute them. That reuse is what makes it efficient (it is reverse-mode automatic differentiation).

## 1. Building blocks and their derivatives

A network is made of three kinds of pieces. To backprop, we need the **local derivative** of each.

### 1.1 The linear (fully-connected) layer

$$\mathbf{z} = W\mathbf{a} + \mathbf{b}, \qquad\text{elementwise:}\quad z_j = \sum_k a_k\,W_{kj} + b_j.$$

Local derivatives (we will reuse these constantly):

$$\frac{\partial z_j}{\partial W_{kj}} = a_k, \qquad \frac{\partial z_j}{\partial b_j} = 1, \qquad \frac{\partial z_j}{\partial a_k} = W_{kj}.$$

### 1.2 Activation functions $g(z)$ and their derivatives

| name | $g(z)$ | $g'(z)$ |
|---|---|---|
| **sigmoid** $\sigma$ | $\dfrac{1}{1+e^{-z}}$ | $\sigma(z)\,(1-\sigma(z)) = a(1-a)$ |
| **tanh** | $\tanh z$ | $1-\tanh^2 z = 1-a^2$ |
| **ReLU** | $\max(0,z)$ | $\mathbb{1}[z>0]$  (1 if $z>0$, else 0) |

*Why these closed forms are so clean:* for sigmoid, $a=\sigma(z)$ gives $a'=a(1-a)$ — the derivative is expressible **using the output itself**, so the forward pass already computed what the backward pass needs. (Derived in Section 3.)

### 1.3 Loss functions and their derivatives

| name | $L$ | $\partial L/\partial \hat{y}$ |
|---|---|---|
| **MSE** (regression) | $\frac1n\sum_i (\hat y_i - y_i)^2$ | $\dfrac{2}{n}(\hat y_i - y_i)$ |
| **Binary cross-entropy** (BCE) | $-\big(y\log\hat y + (1-y)\log(1-\hat y)\big)$ | $\dfrac{\hat y - y}{\hat y(1-\hat y)}$ |
| **Cross-entropy** (multi-class) | $-\sum_i y_i \log \hat y_i$ | $-\,y_i/\hat y_i$ |

Let's put these in code once and reuse them throughout.

In [1]:
import numpy as np
np.random.seed(0)

# ---- activations and their derivatives (all elementwise) ----
def sigmoid(z):      return 1.0 / (1.0 + np.exp(-z))
def sigmoid_d(z):    s = sigmoid(z); return s * (1 - s)

def tanh(z):         return np.tanh(z)
def tanh_d(z):       return 1.0 - np.tanh(z)**2

def relu(z):         return np.maximum(0.0, z)
def relu_d(z):       return (z > 0).astype(z.dtype)

# ---- losses (return a single scalar) and their derivatives wrt the prediction ----
def mse(yhat, y):    return np.mean((yhat - y)**2)
def mse_d(yhat, y):  return 2.0 * (yhat - y) / y.size      # note the 1/n from the mean

eps = 1e-12
def bce(yhat, y):    return -np.mean(y*np.log(yhat+eps) + (1-y)*np.log(1-yhat+eps))
def bce_d(yhat, y):  return (yhat - y) / (yhat*(1-yhat) + eps) / y.size

print("building blocks ready")

building blocks ready


### 1.4 Our verification tool: the **numerical gradient**

Before deriving anything, here is how we'll *prove* every analytic gradient is correct. The definition of a derivative is a limit of a difference quotient. The **centered finite difference** approximates it to high accuracy:

$$\frac{\partial L}{\partial \theta_i} \;\approx\; \frac{L(\theta_i + \varepsilon) - L(\theta_i - \varepsilon)}{2\varepsilon}, \qquad \varepsilon \approx 10^{-6}.$$

We wiggle each parameter up and down, re-run the forward pass, and see how the loss changes. If our hand-derived (analytic) gradient matches this numerical one, the derivation is correct. This is *exactly* the spirit of `03_backprop.ipynb`'s final check against PyTorch autograd — but here we don't even need PyTorch.

In [2]:
def numerical_grad(loss_fn, param, h=1e-6):
    """Centered finite-difference gradient of loss_fn() w.r.t. the array `param`.
    `param` is mutated in place and restored; loss_fn() must read the *current* param."""
    grad = np.zeros_like(param)
    it = np.nditer(param, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        orig = param[idx]
        param[idx] = orig + h; lp = loss_fn()
        param[idx] = orig - h; lm = loss_fn()
        param[idx] = orig                      # restore
        grad[idx] = (lp - lm) / (2*h)
        it.iternext()
    return grad

def rel_err(a, b):
    """Relative error; ~1e-7 means analytic == numerical."""
    return np.max(np.abs(a-b) / (np.maximum(1e-8, np.abs(a)+np.abs(b))))

def check(name, analytic, numeric, tol=1e-5):
    e = rel_err(analytic, numeric)
    print(f"{'OK ' if e < tol else 'BAD'}  {name:<14} rel-err = {e:.2e}")

### 1.5 More activation functions you will meet (LeakyReLU, Softplus, SiLU, GELU)

The three activations above (sigmoid, tanh, ReLU) are enough for Sections 2–8. But modern nets — transformers, diffusion U-Nets, EfficientNets — use a few more. Each is still **elementwise**, so backprop through it is the *same one rule*: multiply the incoming gradient by the local derivative $g'(z)$ (a mask/gate). Only the closed form of $g'$ changes.

| name | $g(z)$ | $g'(z)$ | why it exists / where used |
|---|---|---|---|
| **LeakyReLU** | $\max(\alpha z, z)$ (e.g. $\alpha{=}0.01$) | $\begin{cases}1&z>0\\ \alpha&z\le 0\end{cases}$ | a small negative slope so dead units still learn (fixes "dying ReLU") |
| **Softplus** | $\log(1+e^{z})$ | $\sigma(z)=\dfrac{1}{1+e^{-z}}$ | a smooth ReLU; its derivative is *exactly* the sigmoid |
| **SiLU / Swish** | $z\,\sigma(z)$ | $\sigma(z)\big(1+z(1-\sigma(z))\big)$ | smooth, non-monotone; EfficientNet, many diffusion U-Nets |
| **GELU** | $z\,\Phi(z)$ | $\Phi(z) + z\,\phi(z)$ | the transformer default (BERT, GPT); gates by a Gaussian CDF |

Here $\Phi$ is the standard-normal **CDF** and $\phi$ its **pdf**, with $\Phi(z)=\tfrac12\big(1+\operatorname{erf}(z/\sqrt2)\big)$ and $\phi(z)=\tfrac{1}{\sqrt{2\pi}}e^{-z^2/2}$.

*Deriving SiLU's slope* (product rule on $z\cdot\sigma(z)$, using $\sigma'=\sigma(1-\sigma)$ from Section 3.1):
$$\frac{d}{dz}\big(z\,\sigma(z)\big) = \sigma(z) + z\,\sigma(z)\big(1-\sigma(z)\big) = \sigma(z)\big(1+z(1-\sigma(z))\big).$$

*Deriving GELU's slope* (product rule on $z\cdot\Phi(z)$, with $\Phi'=\phi$): $\;\frac{d}{dz}\big(z\,\Phi(z)\big)=\Phi(z)+z\,\phi(z).$

Below we add them to our toolbox and verify each derivative against a finite difference — exactly the elementwise check, applied to all four at once.

In [ ]:
import math

def leaky_relu(z, a=0.01):   return np.where(z > 0, z, a*z)
def leaky_relu_d(z, a=0.01): return np.where(z > 0, 1.0, a)

def softplus(z):    return np.log1p(np.exp(-np.abs(z))) + np.maximum(z, 0)   # numerically stable
def softplus_d(z):  return sigmoid(z)                                        # exactly the sigmoid

def silu(z):        return z * sigmoid(z)
def silu_d(z):      s = sigmoid(z); return s * (1 + z*(1 - s))

_erf = np.vectorize(math.erf)
def gelu(z):        return 0.5 * z * (1 + _erf(z/np.sqrt(2)))                 # exact (erf) GELU
def gelu_d(z):
    Phi = 0.5*(1 + _erf(z/np.sqrt(2)))                                       # standard-normal CDF
    phi = np.exp(-z**2/2)/np.sqrt(2*np.pi)                                   # standard-normal pdf
    return Phi + z*phi

# ---- verify every derivative elementwise (finite difference) ----
z = np.random.randn(20)
for nm, g, gd in [("leaky", leaky_relu, leaky_relu_d), ("softplus", softplus, softplus_d),
                  ("silu", silu, silu_d), ("gelu", gelu, gelu_d)]:
    fd = np.array([(g(z[i]+1e-6) - g(z[i]-1e-6))/2e-6 for i in range(z.size)])
    check(nm+"'", gd(z), fd)

## 2. The simplest network: a single linear neuron (no hidden layer)

**Model.** One output, no activation, MSE loss. For one example with input $\mathbf{x}\in\mathbb{R}^d$:

$$\hat y = \mathbf{w}^\top \mathbf{x} + b = \sum_{k} w_k x_k + b, \qquad L = (\hat y - y)^2.$$

This is just linear regression, but it already contains forward + backward in miniature.

### 2.1 Forward
Compute $\hat y$, then $L$. Done.

### 2.2 Backward — derive each gradient with the chain rule

We want $\dfrac{\partial L}{\partial w_k}$ and $\dfrac{\partial L}{\partial b}$. Walk the chain $L \leftarrow \hat y \leftarrow (w_k, b)$.

**Step 1 — loss w.r.t. the output:**
$$\frac{\partial L}{\partial \hat y} = \frac{\partial}{\partial \hat y}(\hat y - y)^2 = 2(\hat y - y).$$

**Step 2 — output w.r.t. parameters** (from Section 1.1):
$$\frac{\partial \hat y}{\partial w_k} = x_k, \qquad \frac{\partial \hat y}{\partial b} = 1.$$

**Step 3 — multiply (chain rule):**
$$\boxed{\frac{\partial L}{\partial w_k} = 2(\hat y - y)\,x_k}, \qquad \boxed{\frac{\partial L}{\partial b} = 2(\hat y - y)}.$$

**Batched / vectorized** (a batch $X$ of shape `(n,d)`, targets $\mathbf{y}$ of shape `(n,)`, mean over the batch). Let the error be $\mathbf{e} = \hat{\mathbf{y}} - \mathbf{y}$. Then

$$\frac{\partial L}{\partial \mathbf{w}} = \frac{2}{n}\,X^\top \mathbf{e}, \qquad \frac{\partial L}{\partial b} = \frac{2}{n}\sum_i e_i.$$

Notice the structure already: **gradient of a weight = (input to the layer) ᵀ · (error coming back)**. We will see this *same shape* in every layer of every network below.

In [3]:
# ---- single linear neuron: forward + analytic backward, then numerical check ----
n, d = 16, 4
X = np.random.randn(n, d)
y = np.random.randn(n)                 # regression targets
w = np.random.randn(d)
b = np.random.randn(1)                  # keep b a (1,) array so numerical_grad can wiggle it

def forward():                          # returns scalar loss using CURRENT w, b
    yhat = X @ w + b
    return mse(yhat, y)

# analytic gradients
yhat = X @ w + b
e    = yhat - y
dw   = (2.0/n) * (X.T @ e)
db   = (2.0/n) * e.sum()

# numerical gradients (proof)
check("dL/dw", dw, numerical_grad(forward, w))
check("dL/db", np.array([db]), numerical_grad(forward, b))

OK   dL/dw          rel-err = 2.62e-10
OK   dL/db          rel-err = 1.58e-10


## 3. Adding a nonlinearity: logistic regression (sigmoid + BCE)

Now the output passes through a **sigmoid** (squashing it into $(0,1)$ as a probability) and we use **binary cross-entropy** loss. This single neuron is the classic *logistic regression* classifier, and it reveals the most beautiful cancellation in all of deep learning.

**Model (one example):**
$$z = \mathbf{w}^\top\mathbf{x} + b, \qquad a = \sigma(z) = \frac{1}{1+e^{-z}}, \qquad L = -\big(y\log a + (1-y)\log(1-a)\big).$$

### 3.1 First, derive the sigmoid's derivative

$$\sigma'(z) = \frac{d}{dz}(1+e^{-z})^{-1} = \frac{e^{-z}}{(1+e^{-z})^2} = \frac{1}{1+e^{-z}}\cdot\frac{e^{-z}}{1+e^{-z}} = \sigma(z)\big(1-\sigma(z)\big) = a(1-a).$$

### 3.2 Derive $\partial L/\partial a$

$$\frac{\partial L}{\partial a} = -\frac{y}{a} + \frac{1-y}{1-a} = \frac{-y(1-a) + (1-y)a}{a(1-a)} = \frac{a - y}{a(1-a)}.$$

### 3.3 The magic cancellation: $\partial L/\partial z$

Chain rule, $L \leftarrow a \leftarrow z$:

$$\delta \;=\; \frac{\partial L}{\partial z} = \frac{\partial L}{\partial a}\cdot\frac{\partial a}{\partial z} = \frac{a-y}{a(1-a)}\cdot a(1-a) = \boxed{\,a - y\,}.$$

The $a(1-a)$ terms **cancel exactly**. The error signal at the pre-activation is simply *prediction minus target*. (This is not a coincidence — sigmoid + BCE, and later softmax + cross-entropy, are "matched" pairs designed precisely so this happens. It also keeps gradients healthy: no $a(1-a)$ factor that could vanish.)

### 3.4 Finish with the chain rule to the parameters
$$\frac{\partial L}{\partial w_k} = \delta\,x_k = (a-y)x_k, \qquad \frac{\partial L}{\partial b} = \delta = (a-y).$$

Batched (mean over $n$): with $\mathbf{a}=\sigma(X\mathbf{w}+b)$ and error $\mathbf{e}=\mathbf{a}-\mathbf{y}$,
$$\frac{\partial L}{\partial \mathbf{w}} = \frac1n X^\top\mathbf{e}, \qquad \frac{\partial L}{\partial b} = \frac1n\sum_i e_i.$$

In [4]:
# ---- logistic regression: forward + analytic backward via the (a - y) shortcut ----
n, d = 16, 4
X = np.random.randn(n, d)
y = (np.random.rand(n) > 0.5).astype(float)     # binary 0/1 labels
w = np.random.randn(d) * 0.5
b = np.zeros(1)                                  # (1,) array, see Section 2 note

def forward():
    a = sigmoid(X @ w + b)
    return bce(a, y)

# analytic
a = sigmoid(X @ w + b)
e = (a - y) / n                                  # the famous (a - y), averaged
dw = X.T @ e
db = e.sum()

# numerical (proof)
check("dL/dw", dw, numerical_grad(forward, w))
check("dL/db", np.array([db]), numerical_grad(forward, b))

OK   dL/dw          rel-err = 3.36e-10
OK   dL/db          rel-err = 5.90e-11


## 4. One hidden layer — the `03_backprop.ipynb` network

This is the headline section: the **exact architecture** from the fast.ai course's `03_backprop.ipynb`.

$$\underbrace{\mathbf{x}}_{784} \;\xrightarrow{\;W_1,\mathbf{b}_1\;}\; \mathbf{z}_1 \;\xrightarrow{\text{ReLU}}\; \mathbf{a}_1 \;\xrightarrow{\;W_2,\mathbf{b}_2\;}\; \hat{\mathbf{y}} \;\xrightarrow{\text{MSE}}\; L$$

with hidden width $50$ and a single output. In `03_backprop` the matrices are `w1:(784,50)`, `b1:(50,)`, `w2:(50,1)`, `b2:(1,)`, a batch `X:(n,784)`, and the loss is `mse(out, targ) = ((out[:,0]-targ)**2).mean()`.

### 4.1 Forward (batched, one example per row)

$$Z_1 = X\,W_1 + \mathbf{b}_1, \qquad A_1 = \mathrm{ReLU}(Z_1), \qquad \hat Y = A_1 W_2 + \mathbf{b}_2, \qquad L = \frac1n\sum_i (\hat y_i - y_i)^2.$$

### 4.2 Backward — derive it layer by layer (reverse order)

We propagate the error signal **from the loss backwards**. This is the heart of the course notebook; below, each boxed result is the line of `lin_grad` / `forward_and_backward` it corresponds to.

**The one rule behind every line.** The chain rule says: *to get the gradient of something, take the gradient already arriving at the next thing and multiply by how this thing locally affects it — and if it affects several things, add up all those paths.* Every formula below is just that rule applied once, so I attach to each a one-line **picture** and a tiny **number** to make it concrete.

**(a) Loss → output.** With $\hat y$ of shape `(n,1)` and `out.g` $=\partial L/\partial \hat Y$:
$$\frac{\partial L}{\partial \hat Y} = \frac{2}{n}\,(\hat Y - Y). \qquad\Longleftrightarrow\qquad \texttt{out.g = 2.*diff[:,None]/n}$$
*Picture:* this is simply the slope of the MSE — how much the loss moves when one prediction moves. Bigger error $(\hat Y-Y)$, bigger push; the $2/n$ falls out of differentiating the squared-and-averaged loss. Call this error $G\equiv\partial L/\partial\hat Y$; everything below is built by pushing $G$ backwards.

**(b) Through linear layer 2.** Forward, this layer did $\hat Y = A_1 W_2 + \mathbf b_2$. Picture the weights as **wires**: hidden unit $k$ connects to output $j$ with strength $W_{2,kj}$. With incoming gradient $G$, three gradients come out:

$$\underbrace{\frac{\partial L}{\partial A_1} = G\,W_2^\top}_{\texttt{l2.g = out.g @ w2.T}}, \qquad \underbrace{\frac{\partial L}{\partial W_2} = A_1^\top G}_{\texttt{w2.g = l2.T @ out.g}}, \qquad \underbrace{\frac{\partial L}{\partial \mathbf b_2} = \textstyle\sum_i G_{i\cdot}}_{\texttt{b2.g = out.g.sum(0)}}.$$

- **$\dfrac{\partial L}{\partial A_1} = G\,W_2^\top$ — the signal we pass back.** *Wire picture:* forward, each hidden unit **sends** its value to every output; backward, it **collects blame** from every output it fed, weighted by the same wires. So hidden unit $k$'s blame is $\sum_j G_{ij}W_{2,kj}$ — a weighted sum of the output errors, which is exactly $(G\,W_2^\top)_{ik}$. *Number:* with $W_2=\begin{bmatrix}10&30\\20&40\end{bmatrix}$ and one sample's output error $G=[\,0.5,\;0.1\,]$, hidden unit 0's blame $=0.5\cdot10+0.1\cdot30=8$. **Forward multiply by $W_2$; backward by $W_2^\top$** — the same wires, opposite direction.
- **$\dfrac{\partial L}{\partial W_2} = A_1^\top G$ — the weight gradient (what we actually learn).** *Why the sum:* each weight $W_{2,kj}$ is reused for **every sample** $i$, so its gradient adds up the contributions over the whole batch, $\sum_i A_{1,ik}G_{ij}=(A_1^\top G)_{kj}$. *(Shared over samples ⇒ sum over samples.)*
- **$\dfrac{\partial L}{\partial \mathbf b_2} = \sum_i G_{i\cdot}$ — the bias gradient.** *Why the sum:* the one bias is **added to every row** in the forward pass (broadcast), so backward its gradient is the sum of the per-row gradients. *(Broadcast forward ⇒ sum backward.)*

Each of these is proved element-by-element, with a fuller worked example, in **Section 4.3 Step 2** below.

**(c) Through ReLU.** $A_1=\mathrm{ReLU}(Z_1)$ acts on **each number by itself** (elementwise), so $Z_{1,ik}$ affects only $A_{1,ik}$ — there is **no sum here**, just a multiply by ReLU's local slope:
$$\underbrace{\frac{\partial L}{\partial Z_1} = \frac{\partial L}{\partial A_1}\odot \mathbb{1}[Z_1>0]}_{\texttt{l1.g = (l1>0).float()*l2.g}}.$$
*Gate picture:* ReLU is a **gate**. Forward, it lets a positive number through unchanged (slope $1$) and blocks a negative one to $0$ (slope $0$). Backward, the gradient does the same — it passes straight through where the unit was **on**, and is **killed** where it was **off**. ($\odot$ = multiply element by element; $\mathbb 1[Z_1>0]$ is a mask of $1$s and $0$s.) *Number:* if $Z_1=[\,2,\,-3,\,0.5\,]$ the mask is $[\,1,0,1\,]$, so an incoming $\partial L/\partial A_1=[\,0.4,\,0.6,\,-0.2\,]$ becomes $\partial L/\partial Z_1=[\,0.4,\,0,\,-0.2\,]$. The off unit gets **zero** blame — nudging $Z_1=-3$ keeps its output at $0$, so it cannot change the loss.

**(d) Through linear layer 1.** Exactly the **same three rules as (b)**, now with input $X$ in place of $A_1$ and incoming gradient $\partial L/\partial Z_1$ in place of $G$:
$$\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Z_1}W_1^\top, \qquad \underbrace{\frac{\partial L}{\partial W_1} = X^\top \frac{\partial L}{\partial Z_1}}_{\texttt{w1.g = inp.T @ l1.g}}, \qquad \frac{\partial L}{\partial \mathbf b_1} = \textstyle\sum_i \big(\tfrac{\partial L}{\partial Z_1}\big)_{i\cdot}.$$
Same pictures: the blame at $Z_1$ flows back through the wires $W_1$ to the input; $W_1$'s gradient sums over the batch; $\mathbf b_1$'s gradient sums over the rows. (Here $\partial L/\partial X$ is usually discarded — we do not train the data — but in a deeper net it would be the error handed to the layer before.)

That's the whole algorithm. Notice the **repeating pattern** (the reusable `lin_grad`): for any linear layer with input `inp`, output gradient `out.g`,
> `inp.g = out.g @ w.T` · `w.g = inp.T @ out.g` · `b.g = out.g.sum(0)`.


### 4.3 Deep dive: deriving **every** gradient element-by-element (nothing skipped)

Section 4.2 just **stated** the matrix formulas. If they seemed to appear out of nowhere, this section is for you. We rebuild every one of them **slowly, from scratch**, using only two things:

- the **chain rule**, and
- the **definition of matrix multiplication** $\big((MN)_{rc}=\sum_t M_{rt}N_{tc}\big)$.

Nothing is skipped, and there is a **worked example with real numbers** further down so the symbols stop being abstract.

#### The ONE new idea — this is the entire difficulty of Section 4

In Section 2 (single neuron) each weight affected **exactly one** output, so the chain rule was a single multiplication:
$$\frac{\partial L}{\partial w_k}=\frac{\partial L}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial w_k}.$$

In a real network — **matrices + a whole batch of samples** — a single number usually affects **many** outputs at once:

- one **weight** $W_{2,kj}$ is reused for **every sample** in the batch;
- one **activation** $A_{1,ik}$ feeds into **every output unit** of the next layer.

The rule for that situation is the **multivariable chain rule**: if a quantity $q$ influences the loss through several outputs $o_1,o_2,\dots,o_m$, then you **add up the contribution along every path**:
$$\frac{\partial L}{\partial q}=\sum_{m}\underbrace{\frac{\partial L}{\partial o_m}}_{\text{error at that output}}\;\underbrace{\frac{\partial o_m}{\partial q}}_{\text{local slope}}.$$

That **sum** is the *only* genuinely new thing in Section 4. Every transpose and every `.sum(0)` you are about to see comes from it — nothing more mysterious is happening.

#### Notation — pin the three index letters down once
| symbol | means | range in this net |
|:--:|---|:--:|
| $i$ | which **sample** (a row of the batch) | $1\dots n$ |
| $k$ | which **input** unit of the layer | e.g. $1\dots 50$ |
| $j$ | which **output** unit of the layer | here $1\dots 1$ |

Per-element forward equations (read $\textstyle\sum_k$ as "add over all input units"):
$$z_{1,ij}=\sum_k X_{ik}W_{1,kj}+b_{1,j},\quad a_{1,ij}=\mathrm{ReLU}(z_{1,ij}),\quad \hat y_{ij}=\sum_k A_{1,ik}W_{2,kj}+b_{2,j},\quad L=\frac1n\sum_i(\hat y_{i1}-y_i)^2.$$

#### The 4-step recipe used in every step below
> **(1)** *Which outputs does my variable touch?* &nbsp;→&nbsp; **(2)** *What is the easy local derivative?* &nbsp;→&nbsp; **(3)** *Sum the chain rule over every output it touched.* &nbsp;→&nbsp; **(4)** *Rewrite that sum as a matrix product.*

Keep this recipe visible; Steps 1–4 are just the recipe applied four times.


#### Step 1 — Loss → output &nbsp;&nbsp;(`out.g = 2.*diff[:,None]/n`)

**Why we compute this first.** Backprop works **backwards from the loss**, so the very first thing we need is the error *at the network's output*. Every later gradient is built by taking this signal and multiplying it by local derivatives on the way back. So this is the seed of the whole backward pass.

The loss is an average of per-sample terms, $L=\frac1n\sum_i\ell_i$ with $\ell_i=(\hat y_{i1}-y_i)^2$. Differentiate w.r.t. **one** output $\hat y_{i1}$. It appears in only **one** term of that sum (its own sample's $\ell_i$), so there is no sum to do here:
$$\frac{\partial L}{\partial \hat y_{i1}}=\frac1n\cdot\frac{\partial}{\partial \hat y_{i1}}(\hat y_{i1}-y_i)^2=\frac1n\cdot 2(\hat y_{i1}-y_i)=\frac{2}{n}(\hat y_{i1}-y_i).$$
(Power rule $\frac{d}{dt}t^2=2t$ with $t=\hat y_{i1}-y_i$; the target $y_i$ is a constant, so its derivative is $0$.) Stack all samples into a column vector:
$$\boxed{\;G\;\equiv\;\frac{\partial L}{\partial \hat Y}=\frac{2}{n}(\hat Y-Y)\;}\qquad\Longleftrightarrow\qquad\texttt{out.g = 2.*diff[:,None]/n}.$$
Call this $G$. It has **one column per output unit**, and it is the error signal we now push back through layer 2. (In the example just below we deliberately use *two* output units so this column index isn't trivial.)

---

#### Step 2 — back through linear layer 2 &nbsp;&nbsp;(this is **Section 4.2 part (b)**, with nothing skipped)

The forward pass of this layer was $\;\hat y_{ij}=\sum_k A_{1,ik}W_{2,kj}+b_{2,j}.$ We want **three** gradients, and it really helps to know *why each one exists* before deriving it:

| gradient | what it is **for** |
|---|---|
| $\partial L/\partial W_2$ | a **parameter** gradient — used directly to update the weights: `w2 -= lr * w2.g`. *This is the actual learning.* |
| $\partial L/\partial b_2$ | a **parameter** gradient — used to update the bias `b2`. |
| $\partial L/\partial A_1$ | **not** a parameter. It is the error signal **handed back to the previous layer** (the ReLU, then layer 1). Without it, earlier layers could never learn. Passing this baton backward *is literally* "back-propagation". |

##### A worked example that makes the index $j$ do something
The real `03_backprop` net has a **single** output, which makes the output index $j$ trivial (only one column) and hides what is really going on. So in this example we give the layer **two output units** — then $j$ genuinely ranges over $\{0,1\}$ and you can *see* why a sum over outputs appears. (With one output every $\sum_{j'}$ below has a single term and just drops away — which is exactly why Section 4.2's compact formulas looked sum-free.)

Take $n=2$ samples, hidden width $2$, and now **2 outputs**:
$$A_1=\begin{bmatrix}1&2\\3&4\end{bmatrix},\qquad W_2=\begin{bmatrix}10&30\\20&40\end{bmatrix}\;(\text{rows}=k,\ \text{cols}=j),\qquad b_2=[\,5,\;7\,].$$
$$\Rightarrow\quad \hat Y=A_1W_2+b_2=\begin{bmatrix}55&117\\115&257\end{bmatrix}.$$
Say the error coming back from Step 1 is $G=\dfrac{\partial L}{\partial \hat Y}=\begin{bmatrix}0.5&0.1\\0.3&0.2\end{bmatrix}$ (just illustrative numbers). After each formula we plug these in and check by hand.

---

##### (b1) Gradient w.r.t. the weights, $\partial L/\partial W_2$

Before any algebra, here is the one fact that makes this layer easy: **each output unit owns its own column of $W_2$.** In the forward line for output unit $j'$,
$$\hat y_{i,j'}=\sum_{k'}A_{1,ik'}W_{2,k'j'}+b_{2,j'},$$
*every* weight carries the **same** second index $j'$. So output unit $j'$ uses only **column $j'$** of $W_2$ — it never touches any other column.

Now pick one weight $W_{2,kj}$ (fixed $k$, fixed $j$) and ask the only question the chain rule needs: *which outputs does it touch?* Because it sits in column $j$, it can show up only in outputs of column $j$ — that is, **only when the output's unit index $j'$ equals $j$** — and within that column it appears once for **every sample $i$** (the same weight is reused across the whole batch).

See it in the numbers (the bias adds a constant and holds no weights, so ignore it while hunting):
$$\hat y_{0,0}=\underline{1\cdot10}+2\cdot20,\quad \hat y_{1,0}=\underline{3\cdot10}+4\cdot20,\qquad \hat y_{0,1}=1\cdot30+2\cdot40,\quad \hat y_{1,1}=3\cdot30+4\cdot40.$$
The weight $W_{2,0,0}=10$ appears **only** in the two column-$0$ outputs $\hat y_{0,0},\hat y_{1,0}$ (both samples), and is **absent** from the column-$1$ outputs. So its local derivative is the number sitting next to it where present, and $0$ where absent:
$$\frac{\partial \hat y_{i,j'}}{\partial W_{2,kj}}=\begin{cases}A_{1,ik}&j'=j\quad(\text{weight present, multiplied by }A_{1,ik})\\[3pt]0&j'\neq j\quad(\text{weight not in this output}).\end{cases}$$

Now the chain rule. The weight $W_{2,kj}$ reaches the loss through *several* outputs, so we **add up the contribution through every one** — every $(i,j')$ pair that exists. That "loop over all outputs" is the double sum:
$$\frac{\partial L}{\partial W_{2,kj}}=\sum_i\sum_{j'}\underbrace{\frac{\partial L}{\partial \hat y_{i,j'}}}_{\text{call it }G_{i,j'}}\;\frac{\partial \hat y_{i,j'}}{\partial W_{2,kj}}.$$
($G_{i,j'}$ is just a *name* for the error already sitting at output $(i,j')$, delivered by Step 1 — nothing new.) But we just found the right-hand factor is **zero whenever $j'\neq j$**, so in the inner sum $\sum_{j'}$ only the single term $j'=j$ survives. The inner sum collapses — replace every $j'$ by $j$ and drop $\sum_{j'}$:
$$\frac{\partial L}{\partial W_{2,kj}}=\sum_i G_{i,j}\,A_{1,ik}.$$
Why this particular sum? Two complementary reasons: the **$j'$-sum died** because a weight lives in exactly one output column; the **$i$-sum lived** because that one weight is shared across all samples, so every sample adds a contribution. *(General rule: a shared parameter ⇒ sum its gradient over everything it was shared across.)*

Last, rewrite the sum as a matrix product. Matrix multiplication is **defined** as $(MN)_{kj}=\sum_t M_{kt}N_{tj}$ — the summed index $t$ must be the **column of the left** factor and the **row of the right**. Our summed index is $i$, but in $A_{1,ik}$ that $i$ is the *row*. To slide it into the column slot we transpose: $(A_1^\top)_{ki}=A_{1,ik}$ (transpose only swaps row$\leftrightarrow$column — pure bookkeeping, no new math). Then
$$\sum_i G_{i,j}\,A_{1,ik}=\sum_i (A_1^\top)_{ki}\,G_{i,j}=(A_1^\top G)_{kj}.$$
$$\boxed{\;\frac{\partial L}{\partial W_2}=A_1^\top G\;}\qquad\Longleftrightarrow\qquad\texttt{w2.g = l2.T @ out.g}\qquad\text{(shapes }(\text{hidden},n)(n,\text{out}); \text{real net }(50,n)(n,1)=(50,1)).$$

**✓ Number check.** $\dfrac{\partial L}{\partial W_{2,0,0}}=\sum_i G_{i,0}A_{1,i,0}=0.5\cdot1+0.3\cdot3=1.4$. The $j'=1$ terms contributed nothing. Directly, $A_1^\top G=\begin{bmatrix}1&3\\2&4\end{bmatrix}\begin{bmatrix}0.5&0.1\\0.3&0.2\end{bmatrix}=\begin{bmatrix}1.4&0.7\\2.2&1.0\end{bmatrix}$, and its $(0,0)$ entry is $1.4$. ✓

---

##### (b2) Gradient w.r.t. the bias, $\partial L/\partial b_2$

Exactly the same shape of argument. $b_{2,j}$ is added to output unit $j$ **only** (its own column), once per sample, so it touches outputs $(i,j)$ for all $i$. Adding a constant has slope $1$, so $\dfrac{\partial \hat y_{i,j'}}{\partial b_{2,j}}=1$ if $j'=j$ and $0$ otherwise. The double sum again loses its $j'$ part (only $j'=j$ survives):
$$\frac{\partial L}{\partial b_{2,j}}=\sum_i\sum_{j'}G_{i,j'}\frac{\partial \hat y_{i,j'}}{\partial b_{2,j}}=\sum_i G_{i,j}.$$
That is a sum over the **sample** axis (axis $0$):
$$\boxed{\;\frac{\partial L}{\partial b_2}=\sum_i G_{i\cdot}\;}\qquad\Longleftrightarrow\qquad\texttt{b2.g = out.g.sum(0)}.$$
> **Memory hook:** in the forward pass the bias was **broadcast** (one copy added to every row); so backward its gradient is **summed** over those rows. *Broadcast forward ⇒ sum backward — every time.*

**✓ Number check.** $\sum_i G_{i,0}=0.5+0.3=0.8$ and $\sum_i G_{i,1}=0.1+0.2=0.3$, so $b_2.g=[0.8,\,0.3]$. ✓

---

##### (b3) Gradient w.r.t. the input, $\partial L/\partial A_1$ — the signal we pass back

This one is the **mirror image** of the weights: now the surviving sum is over **outputs**, not samples. Fix one activation $A_{1,ik}$ (sample $i$, hidden unit $k$). Reading the forward line $\hat y_{i,j'}=\sum_{k'}A_{1,ik'}W_{2,k'j'}+b_{2,j'}$, the inner index $k'$ runs over *all* hidden units, so $A_{1,ik}$ appears in the formula for **every output unit $j'$** — but always within its **own sample $i$** (it is never in another sample's row). Its local derivative is the weight multiplying it: $\dfrac{\partial \hat y_{i,j'}}{\partial A_{1,ik}}=W_{2,k,j'}$.

Chain rule over every output $(i'',j')$ — but the derivative is nonzero only when the sample matches, $i''=i$, so the **sample sum collapses** to the single term $i$ while the **output sum $\sum_{j'}$ survives**:
$$\frac{\partial L}{\partial A_{1,ik}}=\sum_{i''}\sum_{j'}G_{i'',j'}\frac{\partial \hat y_{i'',j'}}{\partial A_{1,ik}}=\sum_{j'}G_{i,j'}\,W_{2,k,j'}.$$
(Notice the exact swap from (b1): a *weight* fans out over samples ⇒ sum over $i$; an *activation* fans out over outputs ⇒ sum over $j'$.) Recognize the matrix product with the transpose $(W_2^\top)_{j'k}=W_{2,k,j'}$:
$$\sum_{j'}G_{i,j'}W_{2,k,j'}=\sum_{j'}G_{i,j'}(W_2^\top)_{j'k}=(G\,W_2^\top)_{ik}.$$
$$\boxed{\;\frac{\partial L}{\partial A_1}=G\,W_2^\top\;}\qquad\Longleftrightarrow\qquad\texttt{l2.g = out.g @ w2.T}.$$

**✓ Number check.** $\dfrac{\partial L}{\partial A_{1,0,0}}=\sum_{j'}G_{0,j'}W_{2,0,j'}=0.5\cdot10+0.1\cdot30=8$ — and here the sum genuinely has **two** terms because there are two outputs (with a single output it would be one term). Full result $G\,W_2^\top=\begin{bmatrix}8&14\\9&14\end{bmatrix}$, matching `out.g @ w2.T`. ✓

> **The pattern to memorize — it returns in *every* linear layer.** For input `inp` and incoming gradient `g`:
> &nbsp;&nbsp;`w.g = inp.T @ g` (weights) &nbsp;·&nbsp; `b.g = g.sum(0)` (bias) &nbsp;·&nbsp; `inp.g = g @ w.T` (pass-back).
> And the reason for each transpose/sum is always the same: *a weight is shared over samples (sum over $i$ ⇒ `inp.T`); an activation fans out over outputs (sum over $j$ ⇒ `w.T`); a bias is broadcast over samples (sum over $i$ ⇒ `.sum(0)`).*


#### Step 3 — back through the ReLU &nbsp;&nbsp;(`l1.g = (l1>0).float()*l2.g`)

ReLU treats each number **on its own**: $a_{1,ij}=\mathrm{ReLU}(z_{1,ij})=\max(0,z_{1,ij})$. So each $z_{1,ij}$ touches **exactly one** output ($a_{1,ij}$) — there is **no sum** in this step. We only need its one-variable derivative, which has two cases:
$$z>0:\;\mathrm{ReLU}(z)=z\Rightarrow\mathrm{ReLU}'(z)=1,\qquad\qquad z<0:\;\mathrm{ReLU}(z)=0\Rightarrow\mathrm{ReLU}'(z)=0,$$
i.e. $\mathrm{ReLU}'(z)=\mathbb 1[z>0]$ — it is $1$ where the unit was "on", $0$ where it was "off". Chain rule, element-by-element:
$$\boxed{\;\frac{\partial L}{\partial Z_1}=\underbrace{\frac{\partial L}{\partial A_1}}_{\text{from Step 2 (b3)}}\odot\;\mathbb 1[Z_1>0]\;\equiv\;H\;}\qquad\Longleftrightarrow\qquad\texttt{l1.g = (l1>0).float()*l2.g}.$$
($\odot$ = elementwise multiply.) **Why this behaves like a gate:** the gradient passes straight through wherever the unit was active in the forward pass, and is **zeroed** wherever the unit was off — those off units contributed nothing to the output, so they receive no blame. Call the result $H$: the error signal at $Z_1$, ready for layer 1.

---

#### Step 4 — back through linear layer 1 &nbsp;&nbsp;(`lin_grad(inp, l1, w1, b1)`)

The forward pass here was $z_{1,ij}=\sum_k X_{ik}W_{1,kj}+b_{1,j}$ — **exactly the same shape of equation as layer 2**, only with $X$ where $A_1$ was, and incoming gradient $H$ where $G$ was. So the three derivations are **word-for-word Step 2** with those two substitutions. (If a line looks unfamiliar, reread the matching part of Step 2 — it is the same argument.)

**(d1) weights.** $\dfrac{\partial z_{1,ij}}{\partial W_{1,kj}}=X_{ik}$ (only column $j$, all samples) $\;\Rightarrow\; \dfrac{\partial L}{\partial W_{1,kj}}=\sum_i H_{ij}X_{ik}=(X^\top H)_{kj}\;\Rightarrow\;\boxed{\dfrac{\partial L}{\partial W_1}=X^\top H}$ → `w1.g = inp.T @ l1.g`.

**(d2) bias.** $\dfrac{\partial z_{1,ij}}{\partial b_{1,j}}=1\;\Rightarrow\;\dfrac{\partial L}{\partial b_{1,j}}=\sum_i H_{ij}\;\Rightarrow\;\boxed{\dfrac{\partial L}{\partial b_1}=\sum_i H_{i\cdot}}$ → `b1.g = l1.g.sum(0)`.

**(d3) input.** $\dfrac{\partial z_{1,ij}}{\partial X_{ik}}=W_{1,kj}\;\Rightarrow\;\dfrac{\partial L}{\partial X_{ik}}=\sum_j H_{ij}W_{1,kj}=(H W_1^\top)_{ik}\;\Rightarrow\;\boxed{\dfrac{\partial L}{\partial X}=H W_1^\top}$ → `inp.g = l1.g @ w1.T`.

Here $\partial L/\partial X$ is usually **thrown away** — we do not update the input data. But in a **deeper** network this exact quantity would be the error baton handed to a still-earlier layer. The code computes it anyway so the linear layer is **reusable** as a building block (that is the whole point of `lin_grad`).

---

#### Where each gradient actually goes — the payoff
- **Parameter gradients** $\partial L/\partial W_1,\,\partial L/\partial b_1,\,\partial L/\partial W_2,\,\partial L/\partial b_2$ → handed to the optimizer: `p -= lr * p.g`. **These are the only quantities that change the network.** They are *why* we did any of this.
- **Activation / input gradients** $\partial L/\partial A_1,\,\partial L/\partial Z_1,\,\partial L/\partial X$ → they update **nothing** by themselves. They are the **relay baton** that carries the error from the loss back to each layer, so that layer can compute its own parameter gradients. Each layer multiplies the baton by its local derivative and passes it to the layer before it.

So the entire backward pass is just: **Step 1** (seed the error $G$) → **Step 2** (layer-2 params + baton $A_1$) → **Step 3** (ReLU gate → baton $H$) → **Step 4** (layer-1 params). Three tiny rules — `inp.T @ g`, `g.sum(0)`, `g @ w.T` — plus a ReLU mask, applied right-to-left. The reusable `lin_grad` in Section 4.4 is literally those three lines.


### 4.3.1 The three matrix-calculus facts behind every formula above

Steps 2–4 rest on just **three small facts**. Once these click, the transposes and sums stop looking arbitrary — they become forced.

**Fact 1 — Transpose just swaps "row" and "column".**  $(M^\top)_{rc}=M_{cr}$. No number changes; the same entries are relabelled. Example:
$$M=\begin{bmatrix}1&2&3\\4&5&6\end{bmatrix}\;(2\times3)\qquad\longrightarrow\qquad M^\top=\begin{bmatrix}1&4\\2&5\\3&6\end{bmatrix}\;(3\times2).$$
We reach for a transpose whenever a sum runs over an index that is currently a **row**, because matrix multiply can only sum over an **inner** index (Fact 2). Transposing slides that index into the inner slot — pure relabelling, no new math.

**Fact 2 — Matrix multiply sums over the shared "inner" index.**  By definition
$$(MN)_{rc}=\sum_{t}M_{r,t}\,N_{t,c},$$
where $t$ is **the column of $M$ and the row of $N$** — the two slots that touch in "$M\,N$". The result keeps the **outer** indices: $r$ (row of $M$) and $c$ (column of $N$). So reading any matrix product is one sentence: *"glue the inner indices, sum over them, keep the outer ones."* This is the whole reason a gradient that was "a sum over some index" became "a product with a transpose": the transpose makes the summed index the inner one. Concrete check with $M=\begin{bmatrix}1&2\\3&4\end{bmatrix},\,N=\begin{bmatrix}5\\6\end{bmatrix}$: $(MN)_{0,0}=\sum_t M_{0,t}N_{t,0}=1\cdot5+2\cdot6=17.$

**Fact 3 — Shapes are a free correctness check.**  For $M\,N$ to be legal, $M$'s columns must equal $N$'s rows, and the result is $(\text{rows of }M)\times(\text{cols of }N)$. And a **gradient always has the same shape as the thing it differentiates** (so `w2.g` must match `w2`). Together these almost *force* each formula. Example: $\partial L/\partial W_2$ must be $(\text{hidden}\times\text{out})$. We have only $A_1\,(n\times\text{hidden})$ and $G\,(n\times\text{out})$ to combine; the **one** legal way to get $(\text{hidden}\times\text{out})$ is $A_1^\top G=(\text{hidden}\times n)(n\times\text{out})$. The shapes pick the formula for you — if a candidate has the wrong shape, it is wrong, no calculus needed.

The next code cell makes all three facts concrete in a few lines.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

# ---- Fact 1: transpose just relabels row <-> column (identical numbers) ----
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print("M.shape =", M.shape, "  M.T.shape =", M.T.shape)
print("M.T =\n", M.T)
print("(M.T)[2,1] == M[1,2] ?  ", M.T[2, 1], "==", M[1, 2])

# ---- Fact 2: (M @ N)[r,c] = sum_t M[r,t]*N[t,c]   (sum over INNER index t) ----
M = np.array([[1., 2.],
              [3., 4.]])
N = np.array([[5.],
              [6.]])
manual = np.zeros((M.shape[0], N.shape[1]))
for r in range(M.shape[0]):
    for c in range(N.shape[1]):
        s = 0.0
        for t in range(M.shape[1]):      # t = the inner index we sum over
            s += M[r, t] * N[t, c]
        manual[r, c] = s
print("\nmanual inner-index loop =\n", manual)
print("M @ N                   =\n", M @ N)
assert np.allclose(manual, M @ N)

# ---- Fact 3: a gradient has the SAME shape as its variable; only one
#      contraction yields that shape ----
A1 = np.zeros((2, 5))     # (n=2 samples, hidden=5)
G  = np.zeros((2, 1))     # (n=2 samples, out=1)   == dL/dYhat
print("\nA1.T @ G has shape", (A1.T @ G).shape, " -> must match W2 shape (5, 1). Correct!")

### 4.3.2 The single rule that decides every transpose and every `.sum()`

You never have to *guess* whether a backward step needs a sum or a transpose. **One question decides it:**

> **In the forward pass, how was this quantity copied or reused? Backward, you *sum* the gradient over exactly that copying.**

This is just the chain rule's "add over every path" read in reverse: a quantity that **fanned out** to many places must **collect blame back** from all of them.

| In the forward pass… | …so in the backward pass | appears in code as |
|---|---|---|
| a **weight** is reused for **every sample** | **sum** the gradient over samples ($\sum_i$) | the transpose in `inp.T @ g` |
| an **activation** feeds **every output unit** | **sum** the gradient over outputs ($\sum_j$) | the transpose in `g @ w.T` |
| a **bias** is **broadcast** (added to every row) | **sum** the gradient over rows | `g.sum(0)` |
| a value is used **once** (e.g. passes through ReLU) | **no sum** — just multiply by the local slope | `(z>0)*g` |

**Tiny worked example of "broadcast forward ⇒ sum backward".** Let $f(b)=\sum_{i=1}^{3}(x_i+b)$. The single number $b$ was added $3$ times, so
$$\frac{df}{db}=\underbrace{1}_{i=1}+\underbrace{1}_{i=2}+\underbrace{1}_{i=3}=3=\sum_{i=1}^{3}1.$$
That is *exactly* what `g.sum(0)` does for a bias: add up the gradient coming back from every row the bias was copied into. Same logic, one level up, gives the transposes for weights and activations.

Keep this table beside you and you can write **any** layer's backward pass by inspection — no re-derivation needed.

### 4.3.3 Proof you can run: the three ways must agree

Talk is cheap — let us *prove* the Step-2 formulas numerically. We compute each gradient **three independent ways** and check they give identical numbers:

1. **Literal loops** — a direct transcription of the sums $\sum_i G_{ij}A_{1,ik}$, etc. No matrix tricks at all. This is the raw *definition*.
2. **Matrix formulas** — `A1.T @ G`, `G.sum(0)`, `G @ W2.T`. The *shortcut* we derived.
3. **Numerical gradient** — nudge each entry by a tiny $h$ and watch the loss move: $\dfrac{\partial L}{\partial\theta}\approx\dfrac{L(\theta+h)-L(\theta-h)}{2h}$. This uses **no calculus**, so it is an honest ground truth.

One trick makes method 3 clean. We want the gradient *assuming* the error arriving at the output is exactly our chosen $G$. So pick the scalar loss
$$L=\sum_{i,j}G_{ij}\,\hat y_{ij}=\texttt{(G*Yhat).sum()}\qquad\Longrightarrow\qquad \frac{\partial L}{\partial \hat y_{ij}}=G_{ij}\;\text{(by construction)}.$$
Now finite-differencing this $L$ tests our formulas against reality. **If all three columns of numbers match, the derivation is correct — full stop.**

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

# ---- the exact Step-2 example: n=2 samples, 2 hidden units, 2 outputs ----
A1 = np.array([[1., 2.],
               [3., 4.]])        # (n, hidden)
W2 = np.array([[10., 30.],
               [20., 40.]])      # (hidden, out)
b2 = np.array([5., 7.])          # (out,)
G  = np.array([[0.5, 0.1],       # (n, out) = dL/dYhat  (the error from Step 1)
               [0.3, 0.2]])

def forward(): return A1 @ W2 + b2            # Yhat: (n, out)
def loss():    return np.sum(G * forward())   # scalar L with dL/dYhat == G exactly

n, K = A1.shape       # samples, hidden
J    = W2.shape[1]    # outputs

# ---------- method 1: LITERAL LOOPS (the chain-rule sums, written out) ----------
w2_loop = np.zeros_like(W2)
for k in range(K):
    for j in range(J):
        w2_loop[k, j] = sum(G[i, j] * A1[i, k] for i in range(n))     # sum over samples i

b2_loop = np.array([sum(G[i, j] for i in range(n)) for j in range(J)])  # sum over samples i

a1_loop = np.zeros_like(A1)
for i in range(n):
    for k in range(K):
        a1_loop[i, k] = sum(G[i, j] * W2[k, j] for j in range(J))     # sum over OUTPUTS j

# ---------- method 2: MATRIX FORMULAS ----------
w2_mat, b2_mat, a1_mat = A1.T @ G, G.sum(0), G @ W2.T

# ---------- method 3: NUMERICAL GRADIENT (ground truth, no calculus) ----------
def numerical_grad(param, h=1e-6):
    g = np.zeros_like(param)
    for idx in np.ndindex(param.shape):
        o = param[idx]
        param[idx] = o + h; Lp = loss()
        param[idx] = o - h; Lm = loss()
        param[idx] = o
        g[idx] = (Lp - Lm) / (2 * h)
    return g

w2_num, b2_num, a1_num = numerical_grad(W2), numerical_grad(b2), numerical_grad(A1)

def show(name, loop, mat, num):
    print(f"\n=== {name} ===")
    print("loops     :", np.ravel(loop))
    print("matrix    :", np.ravel(mat))
    print("numerical :", np.ravel(num))
    assert np.allclose(loop, mat) and np.allclose(mat, num, atol=1e-5)
    print("-> all three agree")

show("dL/dW2   (sum over samples i)", w2_loop, w2_mat, w2_num)
show("dL/db2   (sum over samples i)", b2_loop, b2_mat, b2_num)
show("dL/dA1   (sum over outputs j)", a1_loop, a1_mat, a1_num)
print("\nEVERYTHING MATCHES -> the Step-2 formulas ARE exactly the chain-rule sums.")

### 4.3.4 Capstone: the whole tiny network, every gradient auto-checked

Steps 1–4 chain into the complete backward pass. Below is the *entire* Section 4 network — $X\to W_1,b_1\to\mathrm{ReLU}\to W_2,b_2\to\text{MSE}$ — in a few lines, with **all four** parameter gradients verified against the numerical gradient.

Read `backward()` top to bottom: it is literally **Step 1** (seed $G$), then **Step 2** (`w2g, b2g, a1g`), then **Step 3** (the ReLU gate), then **Step 4** (`w1g, b1g`). Nothing else is going on — a deep network is just this block, stacked. The errors print at $\sim\!10^{-10}$, i.e. the analytic formulas match reality to machine precision.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# a tiny instance of the EXACT section-4 net:  n samples, d inputs, h hidden, 1 output
n, d, h = 4, 3, 5
X  = rng.standard_normal((n, d)); y = rng.standard_normal(n)
W1 = rng.standard_normal((d, h)) * 0.5; b1 = np.zeros(h)
W2 = rng.standard_normal((h, 1)) * 0.5; b2 = np.zeros(1)

def forward():
    Z1   = X @ W1 + b1            # linear layer 1
    A1   = np.maximum(Z1, 0.0)    # ReLU
    Yhat = A1 @ W2 + b2           # linear layer 2
    L    = ((Yhat[:, 0] - y) ** 2).mean()   # MSE
    return Z1, A1, Yhat, L

def loss(): return forward()[3]

def backward():
    Z1, A1, Yhat, L = forward()
    G   = 2.0 * (Yhat[:, 0] - y)[:, None] / n   # Step 1: dL/dYhat
    w2g = A1.T @ G                               # Step 2: weights   (sum over samples)
    b2g = G.sum(0)                               #         bias      (sum over samples)
    a1g = G @ W2.T                               #         pass-back (sum over outputs)
    z1g = (Z1 > 0) * a1g                         # Step 3: ReLU gate (no sum)
    w1g = X.T @ z1g                              # Step 4: weights
    b1g = z1g.sum(0)                             #         bias
    return {"W1": w1g, "b1": b1g, "W2": w2g, "b2": b2g}

def numerical_grad(param, h=1e-6):
    g = np.zeros_like(param)
    for idx in np.ndindex(param.shape):
        o = param[idx]
        param[idx] = o + h; Lp = loss()
        param[idx] = o - h; Lm = loss()
        param[idx] = o
        g[idx] = (Lp - Lm) / (2 * h)
    return g

grads = backward()
print("max |analytic - numerical| for every parameter:")
for name, P in [("W1", W1), ("b1", b1), ("W2", W2), ("b2", b2)]:
    print(f"  {name}: {np.abs(grads[name] - numerical_grad(P)).max():.2e}")
print("\nAll errors ~1e-10  =>  the chained Step 1->4 backward pass is correct.")

## Interactive visualizations (Colab)

The cells below are **live and interactive** &mdash; they run here in Colab (or Jupyter), unlike the static HTML export. Use them to *feel* the math from Sections 4.3&ndash;4.5:

1. **3D loss surface + gradient descent** &mdash; rotate the bowl with your mouse, then press **&#9654; Play** to watch the weights roll downhill along the **negative gradient** (the very gradients we derived by hand in Steps 1&ndash;4).
2. **Loss vs step** &mdash; the same descent shown as a falling curve (hover for exact values).
3. **Step-2 gradient calculator** &mdash; drag the entries of the incoming error `G` and watch `w2.g`, `a1.g`, `b2.g` recompute live.

> These use `plotly` and `ipywidgets`, both **preinstalled on Colab** &mdash; nothing to install.

In [ ]:
# === Interactive 3D: loss surface + gradient descent ===
# Drag to rotate the surface; press the Play button to roll the ball downhill.
import numpy as np
import plotly.graph_objects as go

xs = np.array([-2., -1, 0, 1, 2])         # inputs
ys = 1 + 2 * xs                           # targets from the true line  y = 1 + 2x

def loss(w0, w1):
    e = (w0 + w1 * xs) - ys
    return float(np.mean(e ** 2))

def grad(w0, w1):                          # the gradient we derived by hand
    e = (w0 + w1 * xs) - ys
    return np.array([2 * e.mean(), 2 * (e * xs).mean()])

# loss surface over the two weights
w0g = np.linspace(-3, 5, 60)
w1g = np.linspace(-1, 5, 60)
Z = np.array([[loss(a, b) for a in w0g] for b in w1g])

# run gradient descent from a corner and record the path
lr = 0.12
w = np.array([-2.6, 4.6]); path = [w.copy()]
for _ in range(40):
    w = w - lr * grad(*w)                  # w  <-  w - lr * dL/dw
    path.append(w.copy())
P = np.array([[p[0], p[1], loss(*p) + 0.3] for p in path])

surf = go.Surface(x=w0g, y=w1g, z=Z, colorscale="Viridis", opacity=0.9, showscale=False,
                  contours={"z": {"show": True, "usecolormap": True, "project": {"z": True}}})
ball = go.Scatter3d(x=[P[0, 0]], y=[P[0, 1]], z=[P[0, 2]], mode="lines+markers",
                    line=dict(color="red", width=6), marker=dict(size=4, color="red"), name="descent")
frames = [go.Frame(data=[go.Scatter3d(x=P[:k + 1, 0], y=P[:k + 1, 1], z=P[:k + 1, 2],
                                      mode="lines+markers",
                                      line=dict(color="red", width=6),
                                      marker=dict(size=4, color="red"))],
                   traces=[1], name=str(k)) for k in range(len(P))]

fig = go.Figure(data=[surf, ball], frames=frames)
fig.update_layout(
    title="Gradient descent rolling downhill on the MSE loss surface",
    scene=dict(xaxis_title="w0", yaxis_title="w1", zaxis_title="L"),
    margin=dict(l=0, r=0, t=40, b=0),
    updatemenus=[dict(type="buttons", showactive=False, x=0.05, y=0.9,
        buttons=[dict(label="▶ Play", method="animate",
                      args=[None, {"frame": {"duration": 150, "redraw": True}, "fromcurrent": True}])])])
fig.show()
print("converged to w =", np.round(path[-1], 3), " (true answer: [1, 2]); final loss =", round(loss(*path[-1]), 5))

In [ ]:
# === Interactive: watch the loss fall (hover any point for its exact value) ===
import plotly.graph_objects as go

Ls = [loss(p[0], p[1]) for p in path]      # reuses `path` and `loss` from the cell above
fig = go.Figure(go.Scatter(y=Ls, mode="lines+markers", line=dict(color="#5b46e5")))
fig.update_layout(title="Loss vs gradient-descent step",
                  xaxis_title="step", yaxis_title="MSE loss",
                  margin=dict(l=50, r=10, t=40, b=40))
fig.show()

In [ ]:
# === Interactive: the Step-2 gradient calculator (drag the sliders) ===
# Reproduces the exact worked example from Step 2 / 4.3.3:
#   A1 = [[1,2],[3,4]],  W2 = [[10,30],[20,40]].
# Drag the four entries of the incoming error G = dL/dYhat and watch the
# three gradients recompute. Notice: w2.g sums over SAMPLES, a1.g over OUTPUTS.
import numpy as np
from ipywidgets import interact, FloatSlider

A1 = np.array([[1., 2.], [3., 4.]])
W2 = np.array([[10., 30.], [20., 40.]])

def step2_gradients(G00=0.5, G01=0.1, G10=0.3, G11=0.2):
    G = np.array([[G00, G01], [G10, G11]])
    print("G = dL/dYhat =\n", G, "\n")
    print("w2.g = A1.T @ G   (sum over samples i) =\n", A1.T @ G, "\n")
    print("a1.g = G @ W2.T   (sum over outputs j) =\n", G @ W2.T, "\n")
    print("b2.g = G.sum(0)   (sum over samples i) =", G.sum(0))

interact(step2_gradients,
         G00=FloatSlider(0.5, min=-1, max=1, step=0.1, description="G[0,0]"),
         G01=FloatSlider(0.1, min=-1, max=1, step=0.1, description="G[0,1]"),
         G10=FloatSlider(0.3, min=-1, max=1, step=0.1, description="G[1,0]"),
         G11=FloatSlider(0.2, min=-1, max=1, step=0.1, description="G[1,1]"));

### 4.4 Implementation — a direct NumPy port of `03_backprop`

Now the code. Watch each line match a boxed result above; the numerical gradient check at the end is the proof that the derivation is correct.

In [5]:
# ===== A line-for-line NumPy port of 03_backprop.ipynb's forward_and_backward =====
# (PyTorch t.g attribute -> we just return a dict of grads)

def lin(x, w, b):          # the linear layer:  out = x @ w + b
    return x @ w + b

def lin_grad(inp, out_g, w):
    """Backward of a linear layer. Mirrors 03_backprop's lin_grad exactly.
       Given out_g = dL/d(out), returns (inp_g, w_g, b_g)."""
    inp_g = out_g @ w.T            # dL/d(inp)  = out.g @ w.T
    w_g   = inp.T @ out_g          # dL/d(w)    = inp.T @ out.g
    b_g   = out_g.sum(0)           # dL/d(b)    = out.g.sum(0)
    return inp_g, w_g, b_g

def forward_and_backward(X, y, w1, b1, w2, b2):
    # ---- forward:  input -> lin1 -> relu -> lin2 -> mse ----
    z1 = lin(X, w1, b1)            # (n, nh)
    a1 = relu(z1)                  # (n, nh)
    out = lin(a1, w2, b2)          # (n, 1)
    diff = out[:, 0] - y          # (n,)
    loss = (diff**2).mean()

    # ---- backward (reverse order) ----
    out_g = 2.0 * diff[:, None] / X.shape[0]          # (a) dL/d(out), with 1/n
    a1_g, w2_g, b2_g = lin_grad(a1, out_g, w2)        # (b) through lin2
    z1_g = (z1 > 0) * a1_g                            # (c) through ReLU (mask)
    x_g,  w1_g, b1_g = lin_grad(X, z1_g, w1)          # (d) through lin1
    grads = dict(w1=w1_g, b1=b1_g, w2=w2_g, b2=b2_g)
    return loss, grads

# ---- tiny instance with the SAME shape pattern as 784->50->1 (here 4->5->1) ----
n, nin, nh = 12, 4, 5
X = np.random.randn(n, nin)
y = np.random.randn(n)
w1 = np.random.randn(nin, nh) / np.sqrt(nin)
b1 = np.zeros(nh)
w2 = np.random.randn(nh, 1) / np.sqrt(nh)
b2 = np.zeros(1)

loss, g = forward_and_backward(X, y, w1, b1, w2, b2)
print(f"loss = {loss:.4f}")

# ---- numerical gradient check on EVERY parameter (this is the proof) ----
def loss_only():
    z1 = relu(X @ w1 + b1); out = z1 @ w2 + b2
    return ((out[:,0]-y)**2).mean()

check("w1", g['w1'], numerical_grad(loss_only, w1))
check("b1", g['b1'], numerical_grad(loss_only, b1))
check("w2", g['w2'], numerical_grad(loss_only, w2))
check("b2", g['b2'], numerical_grad(loss_only, b2))

loss = 1.7268
OK   w1             rel-err = 2.61e-09
OK   b1             rel-err = 6.90e-10
OK   w2             rel-err = 5.05e-10
OK   b2             rel-err = 2.74e-09


### 4.5 Putting it to work: a tiny training loop (gradient descent)

Backprop only *computes the gradients*. **Learning** = repeatedly nudging each parameter a small step **downhill**:
$$\theta \leftarrow \theta - \eta\,\frac{\partial L}{\partial \theta}\qquad(\eta = \text{learning rate}).$$
Let's confirm the loss actually goes down on a small synthetic regression problem.

In [6]:
# ---- train the 03-style net by gradient descent; watch the loss fall ----
np.random.seed(1)
n, nin, nh = 200, 4, 16
Xtr = np.random.randn(n, nin)
# a nonlinear target so the hidden layer + ReLU are actually needed
ytr = (Xtr[:,0]**2 + np.sin(3*Xtr[:,1]) + Xtr[:,2]*Xtr[:,3])

w1 = np.random.randn(nin, nh)/np.sqrt(nin); b1 = np.zeros(nh)
w2 = np.random.randn(nh, 1)/np.sqrt(nh);    b2 = np.zeros(1)

lr = 0.05
for epoch in range(2000):
    loss, g = forward_and_backward(Xtr, ytr, w1, b1, w2, b2)
    w1 -= lr*g['w1']; b1 -= lr*g['b1']
    w2 -= lr*g['w2']; b2 -= lr*g['b2']
    if epoch % 400 == 0:
        print(f"epoch {epoch:4d}   loss = {loss:.4f}")
print(f"final         loss = {loss:.4f}   (started ~{((ytr-ytr.mean())**2).mean():.4f})")

epoch    0   loss = 4.5088
epoch  400   loss = 0.4154


epoch  800   loss = 0.2971


epoch 1200   loss = 0.2169


epoch 1600   loss = 0.1922


final         loss = 0.1819   (started ~4.0295)


## 5. Generalizing: a deep network with $L$ layers — the four equations of backprop

Now we stop hand-coding two layers and write the **general** algorithm for any depth. Recall the per-layer recipe:
$$\mathbf{z}^{[l]} = W^{[l]}\mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}, \qquad \mathbf{a}^{[l]} = g^{[l]}(\mathbf{z}^{[l]}), \qquad \mathbf{a}^{[0]}=\mathbf{x}.$$

Define the error signal $\boldsymbol{\delta}^{[l]} = \partial L/\partial \mathbf{z}^{[l]}$. Backprop is **four equations** (here in per-example form; the batched/NumPy form follows):

$$
\textbf{(BP1)}\quad \boldsymbol{\delta}^{[L]} = \nabla_{\mathbf a}L \;\odot\; g'^{[L]}(\mathbf{z}^{[L]})
$$
$$
\textbf{(BP2)}\quad \boldsymbol{\delta}^{[l]} = \big(W^{[l+1]\top}\boldsymbol{\delta}^{[l+1]}\big)\;\odot\; g'^{[l]}(\mathbf{z}^{[l]})
$$
$$
\textbf{(BP3)}\quad \frac{\partial L}{\partial W^{[l]}} = \boldsymbol{\delta}^{[l]}\,\mathbf{a}^{[l-1]\top}
$$
$$
\textbf{(BP4)}\quad \frac{\partial L}{\partial \mathbf{b}^{[l]}} = \boldsymbol{\delta}^{[l]}
$$

**Reading them in words:**
- **BP1** — the error at the *last* layer = (how the loss reacts to the output) × (slope of the output's activation). With a matched loss/activation pair (sigmoid+BCE, softmax+CE) this collapses to $\hat{\mathbf y}-\mathbf y$.
- **BP2** — to get the error one layer *earlier*, push the next layer's error back through its weights ($W^{\top}$) and gate it by *this* layer's activation slope. This is the recursive heart: $\boldsymbol\delta^{[l+1]}\to\boldsymbol\delta^{[l]}$.
- **BP3 / BP4** — once you have $\boldsymbol\delta^{[l]}$, the weight gradient is "error **outer-product** the layer's input", and the bias gradient is just the error. (In Section 4 you saw the *exact* same two facts as `w.g = inp.T @ out.g` and `b.g = out.g.sum(0)`.)

> **The two pictures from Section 4.2 are *all* of BP1–BP2.** *Wires* — $W^\top$ spreads each layer's error back to the units that fed it (the $W^{[l+1]\top}\boldsymbol\delta^{[l+1]}$ term). *Gate* — $\odot\,g'(\mathbf z)$ lets that error through only where the activation was responsive (for ReLU, only where the unit was on). BP1 is the same gate applied at the output layer. Everything deeper is just these two moves, repeated.

> **Batched NumPy form** (batch of $n$ rows, `A[l]` shape `(n, n_l)`, `W[l]` shape `(n_{l-1}, n_l)`): with $\Delta^{[l]} = \partial L/\partial Z^{[l]}$ of shape `(n, n_l)`,
> $$\Delta^{[l]} = \big(\Delta^{[l+1]}W^{[l+1]\top}\big)\odot g'(Z^{[l]}),\quad \frac{\partial L}{\partial W^{[l]}}=A^{[l-1]\top}\Delta^{[l]},\quad \frac{\partial L}{\partial \mathbf b^{[l]}}=\textstyle\sum_{\text{rows}}\Delta^{[l]}.$$
> (Transposes flip versus the per-example form because here data is row-wise — the NumPy convention of Section 0.)

### 5.1 A general feed-forward network in NumPy

We store each layer as `(W, b, activation, activation_deriv)`, cache the forward intermediates, then sweep backwards applying BP1–BP4.

In [7]:
class MLP:
    """Feed-forward net of arbitrary depth. Row-wise batches (n, features)."""
    def __init__(self, sizes, acts):
        # sizes = [n_in, n_h1, ..., n_out];  acts = list of (g, g') per layer
        self.W, self.b, self.acts = [], [], acts
        for nin, nout in zip(sizes[:-1], sizes[1:]):
            self.W.append(np.random.randn(nin, nout) / np.sqrt(nin))
            self.b.append(np.zeros(nout))

    def forward(self, X):
        self.Z, self.A = [], [X]            # cache pre-activations and activations
        a = X
        for (g, _), W, b in zip(self.acts, self.W, self.b):
            z = a @ W + b
            a = g(z)
            self.Z.append(z); self.A.append(a)
        return a

    def backward(self, dL_da_out):
        """Given dL/d(output activation), return grads dW, db for every layer."""
        dW = [None]*len(self.W); db = [None]*len(self.b)
        # BP1: start the error at the output layer
        delta = dL_da_out * self.acts[-1][1](self.Z[-1])        # (n, n_L)
        for l in reversed(range(len(self.W))):
            dW[l] = self.A[l].T @ delta                          # BP3
            db[l] = delta.sum(0)                                 # BP4
            if l > 0:                                            # BP2: move one layer back
                delta = (delta @ self.W[l].T) * self.acts[l-1][1](self.Z[l-1])
        return dW, db

# ---- sanity: a 3-hidden-layer net, MSE output, full numerical gradient check ----
np.random.seed(2)
net = MLP([4, 8, 6, 5, 1], acts=[(relu,relu_d),(tanh,tanh_d),(relu,relu_d),(lambda z:z, lambda z:np.ones_like(z))])
n = 10
X = np.random.randn(n, 4); y = np.random.randn(n, 1)

out = net.forward(X)
dW, db = net.backward(mse_d(out, y))               # dL/da_out for MSE

def loss_only(): return mse(net.forward(X), y)
for l in range(len(net.W)):
    check(f"W[{l}]", dW[l], numerical_grad(loss_only, net.W[l]))
    check(f"b[{l}]", db[l], numerical_grad(loss_only, net.b[l]))

OK   W[0]           rel-err = 5.29e-08
OK   b[0]           rel-err = 2.18e-08
OK   W[1]           rel-err = 5.68e-08
OK   b[1]           rel-err = 2.72e-09
OK   W[2]           rel-err = 6.49e-09
OK   b[2]           rel-err = 2.20e-09
OK   W[3]           rel-err = 4.20e-10
OK   b[3]           rel-err = 1.20e-10


## 6. Multi-class classification: softmax + cross-entropy

The most common output head for classification. For an example with $C$ class scores (logits) $\mathbf{z}=(z_1,\dots,z_C)$:

$$a_i = \mathrm{softmax}(\mathbf z)_i = \frac{e^{z_i}}{\sum_{k} e^{z_k}}, \qquad L = -\sum_i y_i \log a_i,$$
where $\mathbf y$ is a **one-hot** vector (1 for the true class, 0 elsewhere).

### 6.1 Derivative of softmax
Using the quotient rule, for the diagonal ($i=j$) and off-diagonal ($i\neq j$) cases this combines into the compact **Jacobian**:
$$\frac{\partial a_i}{\partial z_j} = a_i(\delta_{ij} - a_j), \qquad \delta_{ij}=\begin{cases}1 & i=j\\ 0 & i\neq j\end{cases}.$$

### 6.2 The cancellation again: $\partial L/\partial z_j$
$$\frac{\partial L}{\partial z_j} = \sum_i \frac{\partial L}{\partial a_i}\frac{\partial a_i}{\partial z_j} = \sum_i \Big(-\frac{y_i}{a_i}\Big)\,a_i(\delta_{ij}-a_j) = -\sum_i y_i(\delta_{ij}-a_j) = -y_j + a_j\underbrace{\sum_i y_i}_{=1}.$$

So once more, with a matched loss/activation,
$$\boxed{\;\boldsymbol{\delta} = \frac{\partial L}{\partial \mathbf z} = \mathbf a - \mathbf y\;}$$
— the softmax output minus the one-hot target. This is why frameworks fuse "softmax + cross-entropy" into one op: the gradient is trivially $\mathbf a-\mathbf y$ and numerically stable. (Compare Section 3.3 — same beautiful result, multi-class version.) *Intuition:* the gradient is literally **how wrong the predicted probabilities are**. For the true class $a_j-1<0$, so descent **pushes that logit up**; for every wrong class $a_j-0>0$, so it **pushes those logits down**. The more confident-and-wrong the prediction, the bigger the push.

### 6.3 Putting it together: a deep classifier on a 2-D spiral

In [8]:
def softmax(Z):
    Z = Z - Z.max(1, keepdims=True)          # subtract max per row -> numerical stability
    e = np.exp(Z)
    return e / e.sum(1, keepdims=True)

def make_spiral(points=100, classes=3):
    X = np.zeros((points*classes, 2)); y = np.zeros(points*classes, dtype=int)
    for c in range(classes):
        r = np.linspace(0.0, 1, points)
        t = np.linspace(c*4, (c+1)*4, points) + np.random.randn(points)*0.2
        X[c*points:(c+1)*points] = np.c_[r*np.sin(t), r*np.cos(t)]
        y[c*points:(c+1)*points] = c
    return X, y

np.random.seed(3)
X, y = make_spiral(100, 3)
Y = np.eye(3)[y]                              # one-hot targets, shape (n, 3)

# hidden layers use ReLU/tanh; the FINAL layer is linear (softmax applied in the loss)
clf = MLP([2, 64, 32, 3], acts=[(relu,relu_d),(tanh,tanh_d),(lambda z:z, lambda z:np.ones_like(z))])

lr, n = 1.0, X.shape[0]
for epoch in range(3001):
    logits = clf.forward(X)
    P = softmax(logits)
    loss = -np.sum(Y*np.log(P+1e-12))/n
    # BP1 for softmax+CE collapses to (P - Y)/n ; the final layer is linear so g'=1
    dW, db = clf.backward((P - Y)/n)
    for l in range(len(clf.W)):
        clf.W[l] -= lr*dW[l]; clf.b[l] -= lr*db[l]
    if epoch % 600 == 0:
        acc = (P.argmax(1) == y).mean()
        print(f"epoch {epoch:4d}   loss {loss:.4f}   train-acc {acc:.3f}")
print(f"final train accuracy: {(softmax(clf.forward(X)).argmax(1)==y).mean():.3f}")

epoch    0   loss 1.1562   train-acc 0.313


epoch  600   loss 0.0384   train-acc 0.983


epoch 1200   loss 0.0313   train-acc 0.990


epoch 1800   loss 0.0245   train-acc 0.990


epoch 2400   loss 0.0239   train-acc 0.990


epoch 3000   loss 0.0229   train-acc 0.990
final train accuracy: 0.993


## 7. Convolutional layer — backprop through a convolution

A CNN replaces the dense matrix-multiply with a **convolution** (technically cross-correlation): a small kernel $K$ slides over the input $X$, computing local weighted sums. This shares weights across positions, which is why CNNs are efficient for images / spatial fields.

**Forward (single channel, "valid" cross-correlation).** For input $X$ ($H\times W$) and kernel $K$ ($k_h\times k_w$), the output $Y$ has size $(H-k_h+1)\times(W-k_w+1)$:
$$Y_{i,j} = \sum_{u=0}^{k_h-1}\sum_{v=0}^{k_w-1} X_{i+u,\,j+v}\,K_{u,v}.$$

**Backward.** Let $G = \partial L/\partial Y$ (same shape as $Y$). Two remarkable facts:

1. **Kernel gradient is itself a cross-correlation** of the input with the upstream gradient:
$$\frac{\partial L}{\partial K_{u,v}} = \sum_{i,j} G_{i,j}\,X_{i+u,\,j+v}.$$
*Why:* $K_{u,v}$ is reused at every output position $(i,j)$, so its gradient sums the chain-rule contribution $G_{i,j}\cdot X_{i+u,j+v}$ over all positions (the Section 1.1 "shared parameter ⇒ sum the paths" rule, applied spatially).

2. **Input gradient is a *full* convolution** of $G$ with the **flipped** kernel:
$$\frac{\partial L}{\partial X_{p,q}} = \sum_{u,v} G_{p-u,\,q-v}\,K_{u,v}$$
(treating out-of-range $G$ as zero). *Wire picture:* each input pixel $X_{p,q}$ was used by **several** output positions — every sliding window that covered it — each time through a different kernel entry. Backward, that pixel collects blame from all those outputs, weighted by the kernel entry that touched it; adding up those contributions is exactly a convolution of $G$ with the flipped kernel. Flipping + full-padding is what turns the forward correlation into the correct transposed operation.

Below: a tiny implementation, verified numerically.

In [9]:
def conv2d(X, K):                                   # valid cross-correlation
    H, W = X.shape; kh, kw = K.shape
    oh, ow = H-kh+1, W-kw+1
    Y = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            Y[i, j] = np.sum(X[i:i+kh, j:j+kw] * K)
    return Y

def conv2d_backward(X, K, G):                       # G = dL/dY
    H, W = X.shape; kh, kw = K.shape
    oh, ow = G.shape
    dK = np.zeros_like(K); dX = np.zeros_like(X)
    for i in range(oh):
        for j in range(ow):
            dK += G[i, j] * X[i:i+kh, j:j+kw]       # fact 1: correlation of X with G
            dX[i:i+kh, j:j+kw] += G[i, j] * K       # fact 2: scatter-add flipped kernel
    return dX, dK

np.random.seed(4)
X = np.random.randn(6, 7); K = np.random.randn(3, 3)
target = np.random.randn(4, 5)                      # dummy target to make a scalar loss
def loss_only(): return 0.5*np.sum((conv2d(X, K) - target)**2)

G = conv2d(X, K) - target                           # dL/dY for this 0.5*||.||^2 loss
dX, dK = conv2d_backward(X, K, G)
check("conv dL/dK", dK, numerical_grad(loss_only, K))
check("conv dL/dX", dX, numerical_grad(loss_only, X))

OK   conv dL/dK     rel-err = 2.03e-09
OK   conv dL/dX     rel-err = 1.09e-07


### 7.1 The real thing: a **multi-channel** convolution ($C_\text{in}\to C_\text{out}$)

Section 7 used one input channel and one kernel for clarity. A real conv layer maps a stack of $C_\text{in}$ input feature-maps to $C_\text{out}$ output maps. The kernel is now a 4-D tensor $K$ of shape $(C_\text{out}, C_\text{in}, k_h, k_w)$, and each output channel **sums over all input channels**:

$$Y_{c_o,i,j} = \sum_{c_i=0}^{C_\text{in}-1}\sum_{u=0}^{k_h-1}\sum_{v=0}^{k_w-1} X_{c_i,\,i+u,\,j+v}\;K_{c_o,c_i,u,v}.$$

The backward pass is the Section 7 pair, now carrying the channel indices. With $G=\partial L/\partial Y$ (shape $(C_\text{out},o_h,o_w)$):

$$\frac{\partial L}{\partial K_{c_o,c_i,u,v}} = \sum_{i,j} G_{c_o,i,j}\,X_{c_i,\,i+u,\,j+v}, \qquad \frac{\partial L}{\partial X_{c_i,p,q}} = \sum_{c_o}\sum_{u,v} G_{c_o,\,p-u,\,q-v}\,K_{c_o,c_i,u,v}.$$

It is *literally* Section 7's two facts (correlate-input-with-upstream for the kernel; scatter-the-flipped-kernel for the input) wrapped in an extra sum over channels. Biases (one per output channel) would add $\partial L/\partial b_{c_o}=\sum_{i,j}G_{c_o,i,j}$, exactly the Section 1.1 "broadcast forward ⇒ sum backward" rule.

In [ ]:
# ---- multi-channel valid cross-correlation: forward + backward, numerically checked ----
np.random.seed(0)
Cin, Cout, Hh, Ww, kh, kw = 2, 3, 6, 7, 3, 3
X = np.random.randn(Cin, Hh, Ww)
K = np.random.randn(Cout, Cin, kh, kw)
oh, ow = Hh-kh+1, Ww-kw+1
target = np.random.randn(Cout, oh, ow)

def convNd(X, K):
    Y = np.zeros((Cout, oh, ow))
    for co in range(Cout):
        for i in range(oh):
            for j in range(ow):
                Y[co,i,j] = np.sum(X[:, i:i+kh, j:j+kw] * K[co])   # sum over Cin, kh, kw
    return Y

def convNd_backward(X, K, G):
    dK = np.zeros_like(K); dX = np.zeros_like(X)
    for co in range(Cout):
        for i in range(oh):
            for j in range(ow):
                dK[co]               += G[co,i,j] * X[:, i:i+kh, j:j+kw]   # fact 1, per output channel
                dX[:, i:i+kh, j:j+kw] += G[co,i,j] * K[co]                # fact 2, scatter-add
    return dX, dK

def loss_only(): return 0.5*np.sum((convNd(X, K) - target)**2)
G = convNd(X, K) - target
dX, dK = convNd_backward(X, K, G)
check("convNd dK", dK, numerical_grad(loss_only, K))
check("convNd dX", dX, numerical_grad(loss_only, X))

### 7.2 Pooling layers (max & average) — backprop with **no parameters**

After convolutions, CNNs **downsample** with a pooling layer. It has *no weights*, so there is nothing to learn — but gradients still have to flow *through* it to the conv layers below. Take a $2\times2$ window, stride 2.

**Max pooling.** Forward keeps the largest value in each window: $Y_{i,j}=\max_{(u,v)\in\text{window}}X_{\cdot}$. Backward, only the **winning** element influenced the output, so the upstream gradient is routed entirely to that one position; the other three get $0$:
$$\frac{\partial L}{\partial X_{p,q}} = \begin{cases}G_{i,j} & (p,q)=\arg\max\text{ of window }(i,j)\\ 0 & \text{otherwise.}\end{cases}$$
(This is the Section 1.2 ReLU idea taken to the extreme: a hard *router* — gradient goes to the input that "won".)

**Average pooling.** Forward is the mean of the window, $Y_{i,j}=\tfrac14\sum_{\text{window}}X$. Since each input contributed equally with weight $\tfrac14$, the gradient is shared equally:
$$\frac{\partial L}{\partial X_{p,q}} = \tfrac14\,G_{i,j}\quad\text{for each of the 4 inputs of window }(i,j).$$

In [ ]:
# ---- 2x2 stride-2 pooling: forward + backward, numerically checked ----
np.random.seed(0)
X = np.random.randn(6, 6); target = np.random.randn(3, 3)

def maxpool(X):
    H, W = X.shape; oh, ow = H//2, W//2
    Y = np.zeros((oh, ow)); arg = np.zeros((oh, ow), int)
    for i in range(oh):
        for j in range(ow):
            w = X[2*i:2*i+2, 2*j:2*j+2]
            arg[i,j] = w.argmax(); Y[i,j] = w.max()     # remember WHERE the max was
    return Y, arg

def maxpool_backward(X, arg, G):
    dX = np.zeros_like(X); oh, ow = G.shape
    for i in range(oh):
        for j in range(ow):
            r, c = divmod(arg[i,j], 2)                   # decode argmax position
            dX[2*i+r, 2*j+c] += G[i,j]                   # route gradient to the winner only
    return dX

def avgpool(X):
    H, W = X.shape; oh, ow = H//2, W//2; Y = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow): Y[i,j] = X[2*i:2*i+2, 2*j:2*j+2].mean()
    return Y

def avgpool_backward(X, G):
    dX = np.zeros_like(X); oh, ow = G.shape
    for i in range(oh):
        for j in range(ow): dX[2*i:2*i+2, 2*j:2*j+2] += G[i,j]/4.0   # share equally
    return dX

Y, arg = maxpool(X)
def lo_max(): return 0.5*np.sum((maxpool(X)[0] - target)**2)
check("maxpool dX", maxpool_backward(X, arg, Y-target), numerical_grad(lo_max, X))
def lo_avg(): return 0.5*np.sum((avgpool(X) - target)**2)
check("avgpool dX", avgpool_backward(X, avgpool(X)-target), numerical_grad(lo_avg, X))

## 8. Recurrent network — backpropagation through time (BPTT)

An RNN processes a **sequence** $\mathbf{x}_1,\dots,\mathbf{x}_T$, carrying a hidden state $\mathbf{h}_t$ that summarizes the past. The same weights are reused at **every time step** — so, like a CNN's shared kernel, their gradients **sum over time**.

**Forward (for $t=1,\dots,T$):**
$$\mathbf{h}_t = \tanh\!\big(W_{xh}\mathbf{x}_t + W_{hh}\mathbf{h}_{t-1} + \mathbf{b}_h\big), \qquad \hat{\mathbf y}_t = W_{hy}\mathbf{h}_t + \mathbf{b}_y, \qquad \mathbf{h}_0 = \mathbf 0.$$

**Backward (BPTT).** The total loss $L=\sum_t L_t$. The subtlety: $\mathbf h_t$ influences the loss **both** through $\hat{\mathbf y}_t$ *and* through the next state $\mathbf h_{t+1}$. So its gradient has two incoming streams. Define $\mathbf p_t = \partial L/\partial \mathbf z_t$ where $\mathbf z_t$ is the pre-tanh input. Sweeping $t=T\to1$:

$$\frac{\partial L}{\partial \mathbf h_t} = W_{hy}^\top\,\frac{\partial L_t}{\partial \hat{\mathbf y}_t} \;+\; W_{hh}^\top\,\mathbf p_{t+1}, \qquad \mathbf p_t = \frac{\partial L}{\partial \mathbf h_t}\odot\big(1-\mathbf h_t^2\big)$$

(the $1-\mathbf h_t^2$ is the tanh derivative). Then the **shared** weight gradients accumulate across the whole sequence:

$$\frac{\partial L}{\partial W_{xh}} = \sum_t \mathbf p_t\,\mathbf x_t^\top,\quad \frac{\partial L}{\partial W_{hh}} = \sum_t \mathbf p_t\,\mathbf h_{t-1}^\top,\quad \frac{\partial L}{\partial W_{hy}} = \sum_t \frac{\partial L_t}{\partial \hat{\mathbf y}_t}\,\mathbf h_t^\top.$$

The recursive term $W_{hh}^\top \mathbf p_{t+1}$ chained over many steps is exactly what causes **vanishing/exploding gradients** in long sequences (and motivates LSTMs/GRUs). Tiny verified implementation below.

In [10]:
np.random.seed(5)
T, din, dh, dy = 5, 3, 4, 2                          # seq len, input/hidden/output dims
Xs = [np.random.randn(din) for _ in range(T)]
Ts = [np.random.randn(dy)  for _ in range(T)]        # per-step targets

Wxh = np.random.randn(dh, din)/np.sqrt(din)
Whh = np.random.randn(dh, dh)/np.sqrt(dh)
Why = np.random.randn(dy, dh)/np.sqrt(dh)
bh  = np.zeros(dh); by = np.zeros(dy)

def rnn_forward():
    hs = {-1: np.zeros(dh)}; ys = {}
    loss = 0.0
    for t in range(T):
        hs[t] = np.tanh(Wxh@Xs[t] + Whh@hs[t-1] + bh)
        ys[t] = Why@hs[t] + by
        loss += 0.5*np.sum((ys[t]-Ts[t])**2)
    return loss, hs, ys

def rnn_backward():
    loss, hs, ys = rnn_forward()
    dWxh=np.zeros_like(Wxh); dWhh=np.zeros_like(Whh); dWhy=np.zeros_like(Why)
    dbh=np.zeros_like(bh); dby=np.zeros_like(by)
    dh_next = np.zeros(dh)                            # W_hh^T @ p_{t+1}, starts at 0
    for t in reversed(range(T)):
        dy_t = ys[t]-Ts[t]                            # dL_t/d(yhat_t)
        dWhy += np.outer(dy_t, hs[t]); dby += dy_t
        dh_t = Why.T@dy_t + dh_next                   # two streams into h_t
        p_t  = dh_t * (1 - hs[t]**2)                  # through tanh
        dWxh += np.outer(p_t, Xs[t])
        dWhh += np.outer(p_t, hs[t-1])
        dbh  += p_t
        dh_next = Whh.T @ p_t                         # pass to earlier step
    return dict(Wxh=dWxh, Whh=dWhh, Why=dWhy, bh=dbh, by=dby)

g = rnn_backward()
for name, P in [("Wxh",Wxh),("Whh",Whh),("Why",Why),("bh",bh),("by",by)]:
    check(name, g[name], numerical_grad(lambda: rnn_forward()[0], P))

OK   Wxh            rel-err = 7.07e-09
OK   Whh            rel-err = 2.41e-09
OK   Why            rel-err = 1.64e-09
OK   bh             rel-err = 1.21e-10
OK   by             rel-err = 3.39e-10


## 9. Normalization layers — BatchNorm and LayerNorm

Normalization layers re-center and re-scale activations so deeper nets train faster and more stably. They are everywhere: **BatchNorm** in CNNs, **LayerNorm** in every transformer. Their backward pass is the most-feared derivation in deep learning — because the normalization makes *every* output depend on *every* input in the group (through the shared mean and variance). We'll derive it in full; nothing skipped.

### 9.1 BatchNorm — forward

Normalize **each feature across the batch**. For a batch $X$ of shape `(n, D)`, per feature column $j$:
$$\mu_j = \frac1n\sum_i x_{ij},\qquad \sigma_j^2 = \frac1n\sum_i (x_{ij}-\mu_j)^2,\qquad \hat x_{ij} = \frac{x_{ij}-\mu_j}{\sqrt{\sigma_j^2+\varepsilon}},\qquad y_{ij} = \gamma_j\,\hat x_{ij} + \beta_j.$$
$\gamma,\beta\in\mathbb R^{D}$ are learnable (scale & shift) so the layer can *undo* the normalization if that helps. $\varepsilon$ guards against divide-by-zero.

### 9.2 BatchNorm — backward, derived step by step

Given $\partial L/\partial y \equiv \mathrm dy$ (shape `(n,D)`). Write $\sigma_j^{-1}\equiv(\sigma_j^2+\varepsilon)^{-1/2}$. The learnable params are easy — $\gamma_j,\beta_j$ touch only column $j$, summed over the batch:
$$\frac{\partial L}{\partial \gamma_j} = \sum_i \mathrm dy_{ij}\,\hat x_{ij},\qquad \frac{\partial L}{\partial \beta_j} = \sum_i \mathrm dy_{ij},\qquad \frac{\partial L}{\partial \hat x_{ij}} = \mathrm dy_{ij}\,\gamma_j \;\equiv\; \mathrm d\hat x_{ij}.$$

Now the hard part: $x_{ij}$ reaches $\hat x_{kj}$ **three ways** — directly (numerator of its own $\hat x$), through the shared mean $\mu_j$, and through the shared variance $\sigma_j^2$. Chain-rule each path and add.

**(1) through the variance.** $\hat x_{kj}=(x_{kj}-\mu_j)\,\sigma_j^{-1}$ and $\sigma_j^{-1}=(\sigma_j^2+\varepsilon)^{-1/2}$, so $\partial\sigma_j^{-1}/\partial\sigma_j^2=-\tfrac12(\sigma_j^2+\varepsilon)^{-3/2}=-\tfrac12\sigma_j^{-3}$:
$$\frac{\partial L}{\partial \sigma_j^2} = \sum_k \mathrm d\hat x_{kj}\,(x_{kj}-\mu_j)\cdot\Big(-\tfrac12\sigma_j^{-3}\Big).$$

**(2) through the mean.** $\hat x_{kj}$ depends on $\mu_j$ directly ($-\sigma_j^{-1}$) and via $\sigma_j^2$ (whose dependence on $\mu_j$ sums to zero — a standard simplification), so the surviving term is:
$$\frac{\partial L}{\partial \mu_j} = \sum_k \mathrm d\hat x_{kj}\,(-\sigma_j^{-1}).$$

**(3) collect at $x_{ij}$.** Direct path $\partial\hat x_{ij}/\partial x_{ij}=\sigma_j^{-1}$; through the mean $\partial\mu_j/\partial x_{ij}=\tfrac1n$; through the variance $\partial\sigma_j^2/\partial x_{ij}=\tfrac{2}{n}(x_{ij}-\mu_j)$:
$$\frac{\partial L}{\partial x_{ij}} = \mathrm d\hat x_{ij}\,\sigma_j^{-1} \;+\; \frac{\partial L}{\partial \sigma_j^2}\cdot\frac{2(x_{ij}-\mu_j)}{n} \;+\; \frac{\partial L}{\partial \mu_j}\cdot\frac1n.$$

Substituting (1) and (2) and using $\hat x_{ij}=(x_{ij}-\mu_j)\sigma_j^{-1}$ collapses everything to the famous **compact form** (sums over the batch axis $i$):
$$\boxed{\;\frac{\partial L}{\partial x_{ij}} = \frac{\sigma_j^{-1}}{n}\Big(n\,\mathrm d\hat x_{ij} \;-\; \sum_k \mathrm d\hat x_{kj} \;-\; \hat x_{ij}\sum_k \mathrm d\hat x_{kj}\,\hat x_{kj}\Big).\;}$$
Read it as: *"my own gradient, minus the batch average gradient, minus my share of the gradient that correlates with my normalized value."* The two subtractions are exactly the corrections for the shared $\mu$ and $\sigma^2$.

### 9.3 LayerNorm — the same algebra, axis swapped

LayerNorm normalizes **each example across its features** (so it does not depend on the batch — ideal for sequences/transformers). For row $i$ over $D$ features: $\mu_i=\tfrac1D\sum_j x_{ij}$, $\sigma_i^2=\tfrac1D\sum_j(x_{ij}-\mu_i)^2$, $\hat x_{ij}=(x_{ij}-\mu_i)\sigma_i^{-1}$, $y_{ij}=\gamma_j\hat x_{ij}+\beta_j$. The backward formula is **identical with $n\to D$ and the sums taken over the feature axis $j$**:
$$\frac{\partial L}{\partial x_{ij}} = \frac{\sigma_i^{-1}}{D}\Big(D\,\mathrm d\hat x_{ij} - \sum_k \mathrm d\hat x_{ik} - \hat x_{ij}\sum_k \mathrm d\hat x_{ik}\,\hat x_{ik}\Big),$$
while $\partial L/\partial\gamma_j=\sum_i \mathrm dy_{ij}\hat x_{ij}$ and $\partial L/\partial\beta_j=\sum_i \mathrm dy_{ij}$ still sum over the batch (one $\gamma,\beta$ per feature).

In [ ]:
# ---- BatchNorm & LayerNorm: forward + the compact backward, both numerically checked ----
np.random.seed(0)
n, D, eps = 8, 5, 1e-5
gamma = np.random.randn(D); beta = np.random.randn(D)

# ===== BatchNorm (normalize each feature over the batch axis 0) =====
def bn_forward(X, gamma, beta):
    mu = X.mean(0); var = X.var(0); inv = 1/np.sqrt(var+eps)
    xhat = (X-mu)*inv
    return gamma*xhat + beta, (xhat, inv, gamma)

def bn_backward(dy, cache):
    xhat, inv, gamma = cache; n = dy.shape[0]
    dgamma = (dy*xhat).sum(0); dbeta = dy.sum(0)
    dxhat = dy*gamma
    dx = (inv/n) * (n*dxhat - dxhat.sum(0) - xhat*(dxhat*xhat).sum(0))   # boxed formula
    return dx, dgamma, dbeta

X = np.random.randn(n, D); target = np.random.randn(n, D)
y, cache = bn_forward(X, gamma, beta); dy = y - target
dx, dg, db = bn_backward(dy, cache)
def lo(): return 0.5*np.sum((bn_forward(X, gamma, beta)[0] - target)**2)
check("BN dX", dx, numerical_grad(lo, X))
check("BN dgamma", dg, numerical_grad(lo, gamma))
check("BN dbeta",  db, numerical_grad(lo, beta))

# ===== LayerNorm (normalize each example over the feature axis 1) =====
def ln_forward(X, gamma, beta):
    mu = X.mean(1, keepdims=True); var = X.var(1, keepdims=True); inv = 1/np.sqrt(var+eps)
    xhat = (X-mu)*inv
    return gamma*xhat + beta, (xhat, inv, gamma)

def ln_backward(dy, cache):
    xhat, inv, gamma = cache; D = dy.shape[1]
    dgamma = (dy*xhat).sum(0); dbeta = dy.sum(0)
    dxhat = dy*gamma
    dx = (inv/D) * (D*dxhat - dxhat.sum(1, keepdims=True) - xhat*(dxhat*xhat).sum(1, keepdims=True))
    return dx, dgamma, dbeta

X = np.random.randn(n, D); target = np.random.randn(n, D)
y, cache = ln_forward(X, gamma, beta); dy = y - target
dx, dg, db = ln_backward(dy, cache)
def lo(): return 0.5*np.sum((ln_forward(X, gamma, beta)[0] - target)**2)
check("LN dX", dx, numerical_grad(lo, X))
check("LN dgamma", dg, numerical_grad(lo, gamma))
check("LN dbeta",  db, numerical_grad(lo, beta))

## 10. Dropout — a random gate (and its trivial backward)

Dropout regularizes by randomly **zeroing** activations during training, so the net can't lean on any single unit. We use **inverted dropout**: draw a 0/1 mask with keep-probability $p$ and *rescale by $1/p$* so the expected value is unchanged (then test-time is a plain identity — no rescaling needed).

**Forward (train):** with mask $m_{ij}\sim\mathrm{Bernoulli}(p)/p$,
$$y_{ij} = m_{ij}\,x_{ij}.$$

**Backward.** $y_{ij}$ is just $x_{ij}$ times the constant $m_{ij}$ (the mask is frozen for this forward pass), so by the Section 1.1 product-with-a-constant rule the gradient passes through the **same gate**:
$$\frac{\partial L}{\partial x_{ij}} = m_{ij}\,\frac{\partial L}{\partial y_{ij}}.$$
Dropped units ($m=0$) get zero gradient; kept units pass through scaled by $1/p$. No learnable parameters.

In [ ]:
# ---- inverted dropout: forward + backward, numerically checked (mask held fixed) ----
np.random.seed(0)
n, D, p = 8, 5, 0.7
X = np.random.randn(n, D); target = np.random.randn(n, D)
mask = (np.random.rand(n, D) < p) / p          # Bernoulli(p)/p ; frozen for this pass

def dropout_forward(X):  return X * mask
def lo(): return 0.5*np.sum((dropout_forward(X) - target)**2)
G = dropout_forward(X) - target
dx = G * mask                                   # same gate as forward
check("dropout dX", dx, numerical_grad(lo, X))

## 11. Residual / skip connections — the gradient highway

A residual block computes $\;\mathbf y = \mathbf x + F(\mathbf x)\;$ — the input is **added back** to the output of some sub-network $F$ (two conv layers in a ResNet, attention or an MLP in a transformer). This one addition is why 100-layer nets train at all.

**Backward.** The output depends on $\mathbf x$ through two parallel paths — the identity branch and $F$ — so their gradients **add** (the Section 0 "sum over every path" rule):
$$\frac{\partial L}{\partial \mathbf x} = \underbrace{\frac{\partial L}{\partial \mathbf y}}_{\text{identity branch}} \;+\; \underbrace{J_F^\top\,\frac{\partial L}{\partial \mathbf y}}_{\text{through }F},$$
where $J_F$ is the Jacobian of $F$. The crucial term is the first one: the upstream gradient $\partial L/\partial\mathbf y$ reaches $\mathbf x$ **undiminished**, even if $F$'s own gradient has shrunk to nearly zero. That direct "$+\,\partial L/\partial\mathbf y$" is the *gradient highway* that defeats vanishing gradients in very deep stacks (contrast the multiplicative chains of Section 8 that vanish). Below, $F(\mathbf x)=\mathrm{ReLU}(\mathbf x W)$ as a concrete example.

In [ ]:
# ---- residual block y = x + relu(x @ Wf): forward + backward, numerically checked ----
np.random.seed(0)
n, D = 8, 5
X = np.random.randn(n, D); Wf = np.random.randn(D, D)*0.3; target = np.random.randn(n, D)

def res_forward(X): return X + np.maximum(0, X @ Wf)
def lo(): return 0.5*np.sum((res_forward(X) - target)**2)

G  = res_forward(X) - target            # dL/dy
z  = X @ Wf
dz = G * (z > 0)                        # back through ReLU branch
dX = G + dz @ Wf.T                      # identity branch (G) + branch-through-F
dWf = X.T @ dz
check("residual dX",  dX,  numerical_grad(lo, X))
check("residual dWf", dWf, numerical_grad(lo, Wf))

## 12. Embedding layer — a lookup table, and scatter-add backward

The first layer of every language model maps an integer **token id** to a dense vector. An embedding matrix $W$ has shape $(V, d)$ ($V$ = vocabulary size). Given a list of ids $\mathrm{idx}=(\mathrm{idx}_1,\dots,\mathrm{idx}_n)$, the forward pass is a pure **row lookup**:
$$\mathbf y_i = W_{\mathrm{idx}_i,\,:}\,.$$

This is secretly a matrix multiply by a one-hot matrix $O$ (with $O_{i,\mathrm{idx}_i}=1$): $\,Y=OW$. So by Section 1.1's weight rule $\partial L/\partial W = O^\top\,\partial L/\partial Y$ — and $O^\top(\cdot)$ is exactly a **scatter-add**: each upstream row $\partial L/\partial\mathbf y_i$ is added into row $\mathrm{idx}_i$ of $\partial L/\partial W$. If a token appears several times, its gradient **accumulates** (the shared-parameter rule again):
$$\boxed{\;\frac{\partial L}{\partial W_v} = \sum_{i:\,\mathrm{idx}_i = v} \frac{\partial L}{\partial \mathbf y_i}.\;}$$
There is no gradient w.r.t. the input — the ids are discrete. (Rows for tokens not in the batch get zero gradient and don't update.)

In [ ]:
# ---- embedding lookup: forward + scatter-add backward, numerically checked ----
np.random.seed(0)
V, d = 10, 4
W   = np.random.randn(V, d)
idx = np.array([3, 1, 3, 7, 0])          # note: token 3 appears twice -> its grad accumulates
target = np.random.randn(len(idx), d)

def emb_forward(W): return W[idx]
def lo(): return 0.5*np.sum((emb_forward(W) - target)**2)

G  = emb_forward(W) - target
dW = np.zeros_like(W)
for i, t in enumerate(idx):
    dW[t] += G[i]                         # scatter-add; repeated ids accumulate
check("embedding dW", dW, numerical_grad(lo, W))

## 13. Self-attention (scaled dot-product) — forward and full backward

Attention is the engine of transformers (and of the attention-conditioned diffusion models I'm working through). Every token builds a query, looks up the most relevant tokens by **dot-product similarity**, and reads out a weighted average of their values. We derive its backward pass end to end — it is just the building blocks we already have (linear layers + a row-wise softmax) wired together.

### 13.1 Forward

Input $X$ of shape `(n, d_model)` (n tokens). Three linear projections give queries, keys, values:
$$Q = X W_Q,\quad K = X W_K,\quad V = X W_V\qquad (W_Q,W_K\in\mathbb R^{d_\text{model}\times d_k},\ W_V\in\mathbb R^{d_\text{model}\times d_v}).$$
$$S = \frac{Q K^\top}{\sqrt{d_k}}\ \ (n\times n),\qquad A = \mathrm{softmax}_{\text{row}}(S),\qquad O = A V\ \ (n\times d_v).$$
$S_{ij}$ is "how much token $i$ attends to token $j$"; the $\sqrt{d_k}$ keeps the logits from exploding as $d_k$ grows (so the softmax doesn't saturate). $A$'s rows are probability distributions; $O_i$ is token $i$'s output — a convex combination of value vectors.

### 13.2 Backward — peel it off one op at a time

Start from $\partial L/\partial O\equiv \mathrm dO$ and walk backwards through $O=AV$, the softmax, and $S=QK^\top/\sqrt{d_k}$.

**(a) through $O = AV$** (two Section 1.1 matmul rules): $\;\mathrm dV = A^\top\,\mathrm dO,\qquad \mathrm dA = \mathrm dO\,V^\top.$

**(b) through the row-wise softmax.** Each output row $A_i$ depends only on the corresponding logit row $S_i$, via the softmax Jacobian we already derived in Section 6.1, $\partial A_{ij}/\partial S_{ik}=A_{ij}(\delta_{jk}-A_{ik})$. Contracting $\mathrm dA_i$ against it gives the clean row formula
$$\mathrm dS_{ij} = A_{ij}\Big(\mathrm dA_{ij} - \sum_k \mathrm dA_{ik}A_{ik}\Big) \quad\Longleftrightarrow\quad \mathrm dS = A\odot\big(\mathrm dA - (\mathrm dA\odot A)\mathbf 1\big),$$
i.e. subtract, per row, the $A$-weighted average of $\mathrm dA$ — the exact same "subtract the mean then scale by the prob" shape as softmax+CE in Section 6.

**(c) through the scaling and $S_\text{raw}=QK^\top$:** $\;\mathrm dS_\text{raw} = \mathrm dS/\sqrt{d_k}$, then the two matmul rules once more:
$$\mathrm dQ = \mathrm dS_\text{raw}\,K,\qquad \mathrm dK = \mathrm dS_\text{raw}^\top\,Q.$$

**(d) through the three projections** (Section 1.1 again, with input $X$):
$$\mathrm dW_Q = X^\top \mathrm dQ,\quad \mathrm dW_K = X^\top \mathrm dK,\quad \mathrm dW_V = X^\top \mathrm dV,\qquad \mathrm dX = \mathrm dQ\,W_Q^\top + \mathrm dK\,W_K^\top + \mathrm dV\,W_V^\top.$$
($X$ feeds all three projections, so its three gradient streams **add** — the path-sum rule.)

> **Causal (masked) attention.** A decoder forbids attending to the future by setting $S_{ij}=-\infty$ for $j>i$ *before* the softmax, which makes $A_{ij}=0$ there. Those entries then contribute nothing in the backward pass either ($\mathrm dS$ is zero wherever $A$ is zero) — masking needs no special-case code in the gradient.

In [ ]:
# ---- single-head scaled dot-product attention: forward + full backward, checked ----
np.random.seed(0)
nt, dm, dk, dv = 4, 6, 5, 3
X  = np.random.randn(nt, dm)
Wq = np.random.randn(dm, dk); Wk = np.random.randn(dm, dk); Wv = np.random.randn(dm, dv)
target = np.random.randn(nt, dv)

def softmax_rows(S):
    S = S - S.max(1, keepdims=True); e = np.exp(S); return e / e.sum(1, keepdims=True)

def attn_forward(X, Wq, Wk, Wv):
    Q = X@Wq; K = X@Wk; Vv = X@Wv
    S = Q@K.T / np.sqrt(dk); A = softmax_rows(S); O = A@Vv
    return O, (Q, K, Vv, A)

def lo(): return 0.5*np.sum((attn_forward(X, Wq, Wk, Wv)[0] - target)**2)

O, (Q, K, Vv, A) = attn_forward(X, Wq, Wk, Wv); dO = O - target
dV  = A.T @ dO                                            # (a) through O = A V
dA  = dO @ Vv.T
dS  = A * (dA - (dA*A).sum(1, keepdims=True))             # (b) row-wise softmax Jacobian
dSraw = dS / np.sqrt(dk)                                  # (c) undo the 1/sqrt(dk) scaling
dQ  = dSraw @ K
dK  = dSraw.T @ Q
dWq = X.T @ dQ; dWk = X.T @ dK; dWv = X.T @ dV            # (d) through the projections
dX  = dQ@Wq.T + dK@Wk.T + dV@Wv.T                         # three streams add
for nm, an, pm in [("dWq",dWq,Wq),("dWk",dWk,Wk),("dWv",dWv,Wv),("dX",dX,X)]:
    check("attn "+nm, an, numerical_grad(lo, pm))

## 14. Multi-head attention and the Transformer block

### 14.1 Multi-head attention

One attention pattern is limiting; a transformer runs $H$ of them in **parallel heads**, each on a $d_h=d_\text{model}/H$ slice, then concatenates and mixes with an output projection $W_O$. With $Q=XW_Q,\;K=XW_K,\;V=XW_V$ all of width $d_\text{model}$, head $h$ takes the slice $[\,h d_h:(h{+}1)d_h\,]$:
$$\mathrm{head}_h = \mathrm{softmax}\!\Big(\tfrac{Q_h K_h^\top}{\sqrt{d_h}}\Big)V_h,\qquad O = \big[\mathrm{head}_1\,\|\,\cdots\,\|\,\mathrm{head}_H\big]\,W_O.$$

**Backward** introduces *no new math*: back through $W_O$ ($\mathrm dW_O=\text{concat}^\top \mathrm dO$, $\mathrm d(\text{concat})=\mathrm dO\,W_O^\top$), **slice** that gradient per head, run each slice through the Section 13 single-head backward, then **write each head's $\mathrm dQ_h,\mathrm dK_h,\mathrm dV_h$ back into its column block** of $\mathrm dQ,\mathrm dK,\mathrm dV$ (concatenation forward ⇒ slice the gradient backward). Finally the Section 13.2(d) projection step. We verify the whole thing below.

### 14.2 The Transformer encoder block

A real block wraps attention with the pieces from Section 9 and Section 11 (here in the now-standard **pre-LN** arrangement):
$$\mathbf a = \mathbf x + \mathrm{MHA}\big(\mathrm{LN}_1(\mathbf x)\big),\qquad \mathbf{out} = \mathbf a + \mathrm{FFN}\big(\mathrm{LN}_2(\mathbf a)\big),\qquad \mathrm{FFN}(\mathbf u)=\mathrm{GELU}(\mathbf u W_1+\mathbf b_1)W_2+\mathbf b_2.$$
There is **nothing new to derive**: its backward pass is just our verified pieces composed — LayerNorm (Section 9), multi-head attention (Section 14.1), the GELU MLP (Section 1.5 + Section 5), and **two residual adds (Section 11)** that give two gradient highways straight back to the input. The cell below builds the block and confirms a finite loss with a well-defined $\partial L/\partial X$ — the proof that "a transformer is just these blocks, stacked.\"

In [ ]:
# ---- multi-head attention: explicit backward, numerically checked ----
np.random.seed(0)
H, dmodel, nt = 2, 6, 4
dh = dmodel // H
X  = np.random.randn(nt, dmodel)
Wq = np.random.randn(dmodel, dmodel); Wk = np.random.randn(dmodel, dmodel)
Wv = np.random.randn(dmodel, dmodel); Wo = np.random.randn(dmodel, dmodel)
target = np.random.randn(nt, dmodel)

def mha_forward(X, Wq, Wk, Wv, Wo):
    Q = X@Wq; K = X@Wk; Vv = X@Wv
    outs, cc = [], []
    for h in range(H):
        sl = slice(h*dh, (h+1)*dh)
        q, k, v = Q[:,sl], K[:,sl], Vv[:,sl]
        S = q@k.T/np.sqrt(dh); A = softmax_rows(S); outs.append(A@v); cc.append((q,k,v,A,sl))
    cat = np.concatenate(outs, 1)
    return cat@Wo, (Q, K, Vv, cat, cc)

def lo(): return 0.5*np.sum((mha_forward(X, Wq, Wk, Wv, Wo)[0] - target)**2)

O, (Q, K, Vv, cat, cc) = mha_forward(X, Wq, Wk, Wv, Wo); dO = O - target
dWo = cat.T @ dO; dcat = dO @ Wo.T
dQ = np.zeros_like(Q); dK = np.zeros_like(K); dV = np.zeros_like(Vv)
for (q, k, v, A, sl) in cc:                                   # per head: the Section 13 backward
    dout = dcat[:, sl]
    dV[:,sl] = A.T @ dout
    dA = dout @ v.T
    dS = A*(dA - (dA*A).sum(1, keepdims=True)); dSraw = dS/np.sqrt(dh)
    dQ[:,sl] = dSraw @ k
    dK[:,sl] = dSraw.T @ q
dWq = X.T@dQ; dWk = X.T@dK; dWv = X.T@dV
dX  = dQ@Wq.T + dK@Wk.T + dV@Wv.T
for nm, an, pm in [("dWq",dWq,Wq),("dWk",dWk,Wk),("dWv",dWv,Wv),("dWo",dWo,Wo),("dX",dX,X)]:
    check("MHA "+nm, an, numerical_grad(lo, pm))

# ---- a full pre-LN Transformer block: forward composes everything; dL/dX is well-defined ----
eps = 1e-5
g1 = np.random.randn(dmodel); b1 = np.random.randn(dmodel)
g2 = np.random.randn(dmodel); b2 = np.random.randn(dmodel)
W1 = np.random.randn(dmodel, 8); c1 = np.zeros(8); W2 = np.random.randn(8, dmodel); c2 = np.zeros(dmodel)
def ln(x, g, b):
    mu = x.mean(1, keepdims=True); var = x.var(1, keepdims=True)
    return g*(x-mu)/np.sqrt(var+eps) + b
def transformer_block(X):
    a = X + mha_forward(ln(X, g1, b1), Wq, Wk, Wv, Wo)[0]     # residual 1 (highway)
    h = gelu(ln(a, g2, b2) @ W1 + c1) @ W2 + c2
    return a + h                                              # residual 2 (highway)
def lo_blk(): return 0.5*np.sum((transformer_block(X) - target)**2)
dX_blk = numerical_grad(lo_blk, X)
print(f"transformer block: loss = {lo_blk():.4f}, |dL/dX| = {np.linalg.norm(dX_blk):.4f}  (verified pieces composed)")

## 15. LSTM — gated recurrence and its backward through time

The vanilla RNN of Section 8 vanishes/explodes over long sequences because the same $W_{hh}^\top$ is chained at every step. The **LSTM** fixes this with a protected **cell state** $\mathbf c_t$ that is updated by *additive* gates, so gradients can flow far back in time with little decay.

### 15.1 Forward (per step $t$)

Four gates read the previous hidden state $\mathbf h_{t-1}$ and current input $\mathbf x_t$:
$$\mathbf i_t = \sigma(W_i\mathbf x_t + U_i\mathbf h_{t-1} + \mathbf b_i)\ \ \text{(input)},\qquad \mathbf f_t = \sigma(W_f\mathbf x_t + U_f\mathbf h_{t-1} + \mathbf b_f)\ \ \text{(forget)},$$
$$\mathbf g_t = \tanh(W_g\mathbf x_t + U_g\mathbf h_{t-1} + \mathbf b_g)\ \ \text{(candidate)},\qquad \mathbf o_t = \sigma(W_o\mathbf x_t + U_o\mathbf h_{t-1} + \mathbf b_o)\ \ \text{(output)},$$
$$\boxed{\;\mathbf c_t = \mathbf f_t\odot\mathbf c_{t-1} + \mathbf i_t\odot\mathbf g_t,\qquad \mathbf h_t = \mathbf o_t\odot\tanh(\mathbf c_t).\;}$$
Then a readout $\hat{\mathbf y}_t = W_{hy}\mathbf h_t + \mathbf b_y$ and (here) a per-step squared-error loss.

### 15.2 Backward (BPTT with **two** carries)

Like Section 8, weights are shared across time so their gradients **sum over $t$**. The new feature is a **second** recurrent carry: $\mathbf h_t$ flows into the loss (via $\hat{\mathbf y}_t$) and into step $t{+}1$, *and* $\mathbf c_t$ flows directly into $\mathbf c_{t+1}$. Sweeping $t=T\to1$ with incoming carries $\mathrm dh_\text{next},\mathrm dc_\text{next}$ (both start at $\mathbf 0$):
$$\mathrm dh_t = W_{hy}^\top(\hat{\mathbf y}_t-\mathbf y_t) + \mathrm dh_\text{next},\qquad \mathrm dc_t = \mathrm dh_t\odot\mathbf o_t\odot\big(1-\tanh^2\mathbf c_t\big) + \mathrm dc_\text{next}.$$
Distribute $\mathrm dc_t$ and $\mathrm dh_t$ to the four gate **pre-activations** (using $\sigma'=s(1-s)$ and $\tanh'=1-g^2$):
$$\mathrm do_t = \mathrm dh_t\odot\tanh(\mathbf c_t),\quad \mathrm di_t = \mathrm dc_t\odot\mathbf g_t,\quad \mathrm dg_t = \mathrm dc_t\odot\mathbf i_t,\quad \mathrm df_t = \mathrm dc_t\odot\mathbf c_{t-1},$$
$$\widetilde{\mathrm d i}_t=\mathrm di_t\,\mathbf i_t(1{-}\mathbf i_t),\quad \widetilde{\mathrm d f}_t=\mathrm df_t\,\mathbf f_t(1{-}\mathbf f_t),\quad \widetilde{\mathrm d o}_t=\mathrm do_t\,\mathbf o_t(1{-}\mathbf o_t),\quad \widetilde{\mathrm d g}_t=\mathrm dg_t\,(1{-}\mathbf g_t^2).$$
Each gate is a Section 2-style linear layer, so its weight grads accumulate (outer products) and the two carries pass to the earlier step:
$$\mathrm dW_\bullet \mathrel{+}= \widetilde{\mathrm d\bullet}_t\,\mathbf x_t^\top,\quad \mathrm dU_\bullet \mathrel{+}= \widetilde{\mathrm d\bullet}_t\,\mathbf h_{t-1}^\top,\quad \mathrm db_\bullet \mathrel{+}= \widetilde{\mathrm d\bullet}_t \quad(\bullet\in\{i,f,g,o\}),$$
$$\mathrm dh_\text{next} = U_i^\top\widetilde{\mathrm d i}_t + U_f^\top\widetilde{\mathrm d f}_t + U_g^\top\widetilde{\mathrm d g}_t + U_o^\top\widetilde{\mathrm d o}_t,\qquad \mathrm dc_\text{next} = \mathrm dc_t\odot\mathbf f_t.$$
That last term $\mathrm dc_\text{next}=\mathrm dc_t\odot\mathbf f_t$ is the whole point: with the forget gate near 1, the cell-state gradient is passed back **almost unchanged** — the additive carry that vanilla RNNs lack.

In [ ]:
# ---- LSTM: forward + BPTT, every parameter numerically checked ----
np.random.seed(7)
T, di, dhid, dout = 4, 3, 5, 2
Xs = [np.random.randn(di)   for _ in range(T)]
Ts = [np.random.randn(dout) for _ in range(T)]
def P(*s): return np.random.randn(*s)/np.sqrt(s[-1])
Wi,Ui,bi = P(dhid,di),P(dhid,dhid),np.zeros(dhid)
Wf,Uf,bf = P(dhid,di),P(dhid,dhid),np.zeros(dhid)
Wg,Ug,bg = P(dhid,di),P(dhid,dhid),np.zeros(dhid)
Wo_,Uo,bo= P(dhid,di),P(dhid,dhid),np.zeros(dhid)
Why,by   = P(dout,dhid),np.zeros(dout)

def lstm_forward():
    h = np.zeros(dhid); c = np.zeros(dhid); loss = 0.0
    hs=[h]; cs=[c]; cache=[]
    for t in range(T):
        x = Xs[t]
        i = sigmoid(Wi@x+Ui@h+bi); f = sigmoid(Wf@x+Uf@h+bf)
        g = np.tanh(Wg@x+Ug@h+bg); o = sigmoid(Wo_@x+Uo@h+bo)
        c = f*c + i*g; h = o*np.tanh(c)
        y = Why@h + by; loss += 0.5*np.sum((y-Ts[t])**2)
        cache.append((x,i,f,g,o,c,h,y)); hs.append(h); cs.append(c)
    return loss, cache, hs, cs

def lstm_backward():
    loss, cache, hs, cs = lstm_forward()
    names = dict(Wi=Wi,Ui=Ui,bi=bi,Wf=Wf,Uf=Uf,bf=bf,Wg=Wg,Ug=Ug,bg=bg,Wo_=Wo_,Uo=Uo,bo=bo,Why=Why,by=by)
    G = {k: np.zeros_like(v) for k,v in names.items()}
    dh_next = np.zeros(dhid); dc_next = np.zeros(dhid)
    for t in reversed(range(T)):
        x,i,f,g,o,c,h,y = cache[t]; c_prev = cs[t]; h_prev = hs[t]
        dy = y - Ts[t]
        G['Why'] += np.outer(dy, h); G['by'] += dy
        dh = Why.T@dy + dh_next
        do = dh*np.tanh(c)
        dc = dh*o*(1-np.tanh(c)**2) + dc_next
        di = dc*g; dg = dc*i; df = dc*c_prev
        di_=di*i*(1-i); df_=df*f*(1-f); do_=do*o*(1-o); dg_=dg*(1-g**2)   # through gate nonlinearities
        for nm,dz in [('i',di_),('f',df_),('g',dg_),('o',do_)]:
            W={'i':'Wi','f':'Wf','g':'Wg','o':'Wo_'}[nm]; U={'i':'Ui','f':'Uf','g':'Ug','o':'Uo'}[nm]; b={'i':'bi','f':'bf','g':'bg','o':'bo'}[nm]
            G[W]+=np.outer(dz,x); G[U]+=np.outer(dz,h_prev); G[b]+=dz
        dh_next = Ui.T@di_ + Uf.T@df_ + Ug.T@dg_ + Uo.T@do_
        dc_next = dc*f                                                     # the protected cell-state carry
    return G

g = lstm_backward()
params = dict(Wi=Wi,Ui=Ui,bi=bi,Wf=Wf,Uf=Uf,bf=bf,Wg=Wg,Ug=Ug,bg=bg,Wo_=Wo_,Uo=Uo,bo=bo,Why=Why,by=by)
for nm, Pm in params.items():
    check("LSTM "+nm, g[nm], numerical_grad(lambda: lstm_forward()[0], Pm))

## 16. GRU — the lighter gated cell

The **GRU** keeps the long-memory benefit of the LSTM with fewer parameters: no separate cell state, just two gates — an **update** gate $\mathbf z_t$ (how much of the past to keep) and a **reset** gate $\mathbf r_t$ (how much past to use when proposing new content).

**Forward (per step $t$):**
$$\mathbf z_t = \sigma(W_z\mathbf x_t + U_z\mathbf h_{t-1} + \mathbf b_z),\qquad \mathbf r_t = \sigma(W_r\mathbf x_t + U_r\mathbf h_{t-1} + \mathbf b_r),$$
$$\tilde{\mathbf h}_t = \tanh\!\big(W_h\mathbf x_t + U_h(\mathbf r_t\odot\mathbf h_{t-1}) + \mathbf b_h\big),\qquad \boxed{\;\mathbf h_t = (1-\mathbf z_t)\odot\mathbf h_{t-1} + \mathbf z_t\odot\tilde{\mathbf h}_t.\;}$$

**Backward (BPTT).** The interpolation $\mathbf h_t=(1{-}\mathbf z_t)\mathbf h_{t-1}+\mathbf z_t\tilde{\mathbf h}_t$ is the engine: with incoming $\mathrm dh_t=W_{hy}^\top(\hat{\mathbf y}_t-\mathbf y_t)+\mathrm dh_\text{next}$,
$$\mathrm dz_t = \mathrm dh_t\odot(\tilde{\mathbf h}_t-\mathbf h_{t-1}),\qquad \mathrm d\tilde{\mathbf h}_t = \mathrm dh_t\odot\mathbf z_t,\qquad \mathrm dh_{t-1}\mathrel{+}= \mathrm dh_t\odot(1-\mathbf z_t)\ \ \text{(direct path)}.$$
Through $\tilde{\mathbf h}$'s tanh, $\widetilde{\mathrm dh}_t=\mathrm d\tilde{\mathbf h}_t\odot(1-\tilde{\mathbf h}_t^2)$, which feeds $W_h,U_h$ and — because the reset gate multiplies $\mathbf h_{t-1}$ inside — sends $\mathrm d(\mathbf r\odot\mathbf h_{t-1})=U_h^\top\widetilde{\mathrm dh}_t$ to both $\mathbf r_t$ (giving $\mathrm dr_t$) and back to $\mathbf h_{t-1}$. After pushing $\mathbf z,\mathbf r$ through their sigmoids ($\widetilde{\mathrm dz},\widetilde{\mathrm dr}$), the weight grads accumulate as outer products and $\mathrm dh_\text{next}=\mathrm dh_{t-1}$ collects every path back to the previous step. Verified below — note there is only **one** carry (no cell state).

In [ ]:
# ---- GRU: forward + BPTT, every parameter numerically checked ----
np.random.seed(11)
Wz,Uz,bz   = P(dhid,di),P(dhid,dhid),np.zeros(dhid)
Wr,Ur,br   = P(dhid,di),P(dhid,dhid),np.zeros(dhid)
Wh,Uh,bhh  = P(dhid,di),P(dhid,dhid),np.zeros(dhid)
Why2,by2   = P(dout,dhid),np.zeros(dout)

def gru_forward():
    h = np.zeros(dhid); loss = 0.0; hs=[h]; cache=[]
    for t in range(T):
        x = Xs[t]
        z = sigmoid(Wz@x+Uz@h+bz); r = sigmoid(Wr@x+Ur@h+br)
        ht = np.tanh(Wh@x + Uh@(r*h) + bhh)
        hnew = (1-z)*h + z*ht
        y = Why2@hnew + by2; loss += 0.5*np.sum((y-Ts[t])**2)
        cache.append((x,z,r,ht,h,hnew,y)); hs.append(hnew); h = hnew
    return loss, cache, hs

def gru_backward():
    loss, cache, hs = gru_forward()
    names = dict(Wz=Wz,Uz=Uz,bz=bz,Wr=Wr,Ur=Ur,br=br,Wh=Wh,Uh=Uh,bhh=bhh,Why2=Why2,by2=by2)
    G = {k: np.zeros_like(v) for k,v in names.items()}
    dh_next = np.zeros(dhid)
    for t in reversed(range(T)):
        x,z,r,ht,h_prev,hnew,y = cache[t]
        dy = y - Ts[t]
        G['Why2'] += np.outer(dy, hnew); G['by2'] += dy
        dh = Why2.T@dy + dh_next
        dz = dh*(ht - h_prev); dht = dh*z; dh_prev = dh*(1-z)             # interpolation split
        dz_ = dz*z*(1-z); dht_ = dht*(1-ht**2)
        G['Wh']+=np.outer(dht_,x); G['Uh']+=np.outer(dht_, r*h_prev); G['bhh']+=dht_
        drh = Uh.T@dht_                                                   # d wrt (r * h_prev)
        dr = drh*h_prev; dh_prev += drh*r
        dr_ = dr*r*(1-r)
        G['Wz']+=np.outer(dz_,x); G['Uz']+=np.outer(dz_,h_prev); G['bz']+=dz_
        G['Wr']+=np.outer(dr_,x); G['Ur']+=np.outer(dr_,h_prev); G['br']+=dr_
        dh_prev += Uz.T@dz_ + Ur.T@dr_
        dh_next = dh_prev
    return G

g = gru_backward()
params = dict(Wz=Wz,Uz=Uz,bz=bz,Wr=Wr,Ur=Ur,br=br,Wh=Wh,Uh=Uh,bhh=bhh,Why2=Why2,by2=by2)
for nm, Pm in params.items():
    check("GRU "+nm, g[nm], numerical_grad(lambda: gru_forward()[0], Pm))

## 9. One-page cheat-sheet

**Forward (layer $l$):**  $\;\mathbf z^{[l]} = W^{[l]}\mathbf a^{[l-1]}+\mathbf b^{[l]}$, $\;\;\mathbf a^{[l]} = g(\mathbf z^{[l]})$.

**Backward — the four equations:**

| | equation | "in words" |
|---|---|---|
| BP1 | $\boldsymbol\delta^{[L]} = \nabla_{\mathbf a}L \odot g'(\mathbf z^{[L]})$ | error at the output |
| BP2 | $\boldsymbol\delta^{[l]} = (W^{[l+1]\top}\boldsymbol\delta^{[l+1]}) \odot g'(\mathbf z^{[l]})$ | push error back one layer |
| BP3 | $\partial L/\partial W^{[l]} = \boldsymbol\delta^{[l]}\,\mathbf a^{[l-1]\top}$ | weight grad = error ⊗ input |
| BP4 | $\partial L/\partial \mathbf b^{[l]} = \boldsymbol\delta^{[l]}$ | bias grad = error |

**Matched pairs that make BP1 collapse to $\hat{\mathbf y}-\mathbf y$:** sigmoid+BCE (binary), softmax+cross-entropy (multi-class), linear+MSE (regression, up to the constant $2/n$).

**The single repeating motif** (the reusable `lin_grad` from `03_backprop`):
> `inp.g = out.g @ W.T` · `W.g = inp.T @ out.g` · `b.g = out.g.sum(0)`

**Activation derivatives:** $\sigma' = a(1-a)$; $\;\tanh' = 1-a^2$; $\;\mathrm{ReLU}' = \mathbb 1[z>0]$.

**Shared weights ⇒ sum the gradient** over wherever they're reused: over the batch (dense), over spatial positions (CNN), over time steps (RNN/BPTT).

**Gradient descent** turns gradients into learning: $\theta \leftarrow \theta - \eta\,\partial L/\partial\theta$.

---

### A bridge to my own work (climate / Earth-observation)

The exact same backprop machinery trains every model I care about in climate AI: a **U-Net** for statistical downscaling or LST super-resolution is just Section 5's deep net with convolutional layers (Section 7) and skip connections; a **diffusion weather emulator** is trained by backprop through a noise-prediction loss; **ConvLSTM** nowcasting stacks Section 7 (conv) inside Section 8 (BPTT). Understanding Sections 4–5 by hand is exactly what lets me debug *why* a downscaling net's gradients vanish, or why a custom physics-informed loss won't train — I can see it's BP1/BP2, not magic.

---

*That's the whole of backpropagation: three small matrix rules — `inp.T @ g`, `g.sum(0)`, `g @ w.T` — plus the chain rule, stacked. I derived every line by hand and watched the finite-difference checks pass, and that is when it stopped feeling like magic.*